In [1]:
import  numpy as np
import  matplotlib.pyplot as plt
import scipy.stats as stats
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [2]:
def add_cross_sectional_zscore(X, base_features, suffix='_ZSCORE'):
    """
    For each feature in base_features, compute its z-score relative to the
    cross-section of all allocations present at the same TS.
    z = (x - mean(x at this TS)) / std(x at this TS)

    This isolates relative strength on a given day, independent of that
    day's overall market regime (volatile vs calmf days), which raw values
    conflate.
    """
    new_cols = []
    for feat in base_features:
        ts_mean = X.groupby('TS')[feat].transform('mean')
        ts_std = X.groupby('TS')[feat].transform('std')

        new_col = f'{feat}{suffix}'
        # avoid division by zero on days where the feature is constant across the universe
        X[new_col] = np.where(ts_std > 0, (X[feat] - ts_mean) / ts_std, 0.0)
        new_cols.append(new_col)

    return X, new_cols
def compute_bayesian_alloc_rate(train_targets, train_allocs, prior_strength=50):
    """
    Beta-Binomial posterior mean for each allocation's positive-rate.
    prior_strength (alpha0 + beta0) controls how much weight the prior carries,
    same role as 'k' in your earlier shrinkage estimator -- but here it's an
    explicit, principled Bayesian prior, not an ad-hoc formula.
    """
    global_rate = (np.sign(train_targets) > 0).mean()

    # prior: Beta(alpha0, beta0) centered on the global rate, with `prior_strength`
    # total pseudo-observations
    alpha0 = global_rate * prior_strength
    beta0 = (1 - global_rate) * prior_strength

    df = pd.DataFrame({'ALLOCATION': train_allocs, 'target': train_targets.values})
    grouped = df.groupby('ALLOCATION')['target']

    successes = grouped.apply(lambda s: (np.sign(s) > 0).sum())
    n = grouped.size()

    # posterior: Beta(alpha0 + successes, beta0 + failures)
    posterior_alpha = alpha0 + successes
    posterior_beta = beta0 + (n - successes)

    posterior_mean = posterior_alpha / (posterior_alpha + posterior_beta)

    # bonus: a genuine 90% credible interval per allocation, not just a point estimate
    ci_lower = stats.beta.ppf(0.05, posterior_alpha, posterior_beta)
    ci_upper = stats.beta.ppf(0.95, posterior_alpha, posterior_beta)

    return posterior_mean, ci_lower, ci_upper, global_rate
def build_transition_feature(X, n_bins=4):
    """Builds the raw (unbinned) current-state array for every row, using its
    own 20-day RET sequence. This part uses NO target info, so it's safe to
    compute once outside the fold loop."""
    ret_cols_ordered = [f'RET_{i}' for i in range(20, 0, -1)]
    ret_seq = X[ret_cols_ordered].to_numpy()
    return ret_seq  # raw sequence; binning happens per-fold using fold-specific edges

def compute_fold_transition_lookup(ret_seq_train, n_bins=4):
    """Fits bin edges AND the transition matrix using only this fold's training rows.
    Returns a function that maps a row's CURRENT (most recent) state to
    P(next day = highest bin | current state) -- the 'P(next positive-extreme)' feature."""
    clean_values = ret_seq_train[~np.isnan(ret_seq_train)]
    edges = np.quantile(clean_values, np.linspace(0, 1, n_bins + 1))

    def get_states(ret_seq):
        ret_seq_clean = np.nan_to_num(ret_seq, nan=0.0)
        return np.digitize(ret_seq_clean, edges[1:-1])

    states_train = get_states(ret_seq_train)

    transition_counts = np.zeros((n_bins, n_bins))
    for row in states_train:
        for t in range(len(row) - 1):
            transition_counts[row[t], row[t + 1]] += 1
    row_sums = transition_counts.sum(axis=1, keepdims=True)
    transition_matrix = transition_counts / np.where(row_sums == 0, 1, row_sums)

    # "current state" = state of the most recent day (RET_1, last column after our ordering)
    def lookup_feature(ret_seq_query):
        query_states = get_states(ret_seq_query)
        current_state = query_states[:, -1]  # last column = most recent day (RET_1)
        # P(next day's state == highest bin | current state)
        return transition_matrix[current_state, n_bins - 1]

    return lookup_feature
X_train = pd.read_csv('../data/X_train.csv', index_col='ROW_ID')
X_test = pd.read_csv('../data/X_test.csv', index_col='ROW_ID')
y_train = pd.read_csv('../data/y_train.csv', index_col='ROW_ID')
y_train_bin = (y_train['target'] > 0).astype(int)
sample_submission = pd.read_csv('../data/sample_submission.csv', index_col='ROW_ID')

print("Train TS:", X_train['TS'].nunique())
print("Test TS:", X_test['TS'].nunique())

train_ts = set(X_train['TS'].unique())
test_ts = set(X_test['TS'].unique())

print("Shared TS:", len(train_ts & test_ts))
print("Test-only TS:", len(test_ts - train_ts))
print("Train-only TS:", len(train_ts - test_ts))
# ============================================================
# BASIC FEATURES
# ============================================================

RET_features = [f'RET_{i}' for i in range(1, 21)]

SIGNED_VOLUME_features = [
    f'SIGNED_VOLUME_{i}' for i in range(1, 21)
]

TURNOVER_features = [
    'MEDIAN_DAILY_TURNOVER'
]


# ============================================================
# COMMENTED OUT:
# CROSS-SECTIONAL Z-SCORES
#
# We are removing these for this experiment.
# ============================================================

x_train, RET_zscore_features = add_cross_sectional_zscore(
     X_train,
     RET_features
 )

X_test, _ = add_cross_sectional_zscore(
     X_test,
   RET_features
 )


# ============================================================
# COMMENTED OUT:
# CROSS-SECTIONAL CURRENT-DAY MARKET AVERAGE
# ============================================================

X_train['CROSS_SECTIONAL_AVG_RET1'] = (
     X_train.groupby('TS')['RET_1'].transform('mean')
)

X_test['CROSS_SECTIONAL_AVG_RET1'] = (
   X_test.groupby('TS')['RET_1'].transform('mean')
 )


# ============================================================
# INDIVIDUAL ALLOCATION PERFORMANCE FEATURES
#
# KEEP THESE.
# ============================================================

for i in [3, 5, 10, 15, 20]:

    X_train[f'AVERAGE_PERF_{i}'] = (
        X_train[RET_features[:i]].mean(1)
    )

    X_test[f'AVERAGE_PERF_{i}'] = (
        X_test[RET_features[:i]].mean(1)
    )


# ============================================================
# COMMENTED OUT:
# CROSS-SECTIONAL AVERAGE PERFORMANCE
# ============================================================

# for i in [3, 5, 10, 15, 20]:

#     X_train[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = (
#         X_train.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
#     )

#     X_test[f'ALLOCATIONS_AVERAGE_PERF_{i}'] = (
#         X_test.groupby('TS')[f'AVERAGE_PERF_{i}'].transform('mean')
#     )


# ============================================================
# INDIVIDUAL PERFORMANCE STD
#
# KEEP.
# ============================================================

X_train['STD_PERF_20'] = (
    X_train[RET_features[:20]].std(1)
)

X_test['STD_PERF_20'] = (
    X_test[RET_features[:20]].std(1)
)


# ============================================================
# COMMENTED OUT:
# CROSS-SECTIONAL PERFORMANCE STD
# ============================================================

# X_train['ALLOCATIONS_STD_PERF_20'] = (
#     X_train.groupby('TS')['STD_PERF_20'].transform('mean')
# )

# X_test['ALLOCATIONS_STD_PERF_20'] = (
#     X_test.groupby('TS')['STD_PERF_20'].transform('mean')
# )


# ============================================================
# BASE FEATURE LIST
# ============================================================

features = RET_features + SIGNED_VOLUME_features + TURNOVER_features

features = features + [
    f'AVERAGE_PERF_{i}' for i in [3, 5, 10, 15, 20]
]

features = features + [
    'STD_PERF_20'
]


# ============================================================
# COMMENTED OUT:
# CROSS-SECTIONAL FEATURES
# ============================================================

# features = features + [
#     f'ALLOCATIONS_AVERAGE_PERF_{i}'
#     for i in [3, 5, 10, 15, 20]
# ]

features = features + [
     'ALLOCATIONS_STD_PERF_20'
 ]

features = features + RET_zscore_features

minimal_features = features + [
    'CROSS_SECTIONAL_AVG_RET1'
 ]

minimal_features = features.copy()


# ============================================================
# VOLUME FEATURES
#
# KEEP INDIVIDUAL VOLUME FEATURES.
# ============================================================

SIGNED_VOLUME_features_clean = [
    f'SIGNED_VOLUME_{i}' for i in range(2, 21)
]

for i in [3, 5, 10, 15, 20]:

    cols = SIGNED_VOLUME_features_clean[:i-1]

    X_train[f'AVERAGE_VOLUME_{i}'] = (
        X_train[cols].mean(1)
    )

    X_test[f'AVERAGE_VOLUME_{i}'] = (
        X_test[cols].mean(1)
    )


# ============================================================
# COMMENTED OUT:
# CROSS-SECTIONAL AVERAGE VOLUME
# ============================================================

# for i in [3, 5, 10, 15, 20]:

#     X_train[f'ALLOCATIONS_AVERAGE_VOLUME_{i}'] = (
#         X_train.groupby('TS')[f'AVERAGE_VOLUME_{i}'].transform('mean')
#     )

#     X_test[f'ALLOCATIONS_AVERAGE_VOLUME_{i}'] = (
#         X_test.groupby('TS')[f'AVERAGE_VOLUME_{i}'].transform('mean')
#     )


# ============================================================
# INDIVIDUAL VOLUME STD
#
# KEEP.
# ============================================================

cols = SIGNED_VOLUME_features_clean[:19]

X_train['STD_VOLUME_20'] = (
    X_train[cols].std(1)
)

X_test['STD_VOLUME_20'] = (
    X_test[cols].std(1)
)


# ============================================================
# COMMENTED OUT:
# CROSS-SECTIONAL VOLUME STD
# ============================================================

# X_train['ALLOCATIONS_STD_VOLUME_20'] = (
#     X_train.groupby('TS')['STD_VOLUME_20'].transform('mean')
# )

# X_test['ALLOCATIONS_STD_VOLUME_20'] = (
#     X_test.groupby('TS')['STD_VOLUME_20'].transform('mean')
# )


# ============================================================
# ADD INDIVIDUAL VOLUME FEATURES
# ============================================================

volume_features = (
    [f'AVERAGE_VOLUME_{i}' for i in [3, 5, 10, 15, 20]]
    + ['STD_VOLUME_20']
)

features = features + volume_features

minimal_features = features.copy()


# ============================================================
# KEEP THESE INTERACTION FEATURES
# ============================================================

for df in [X_train, X_test]:

    df['RET1_X_SV1'] = (
        df['RET_1'] * df['SIGNED_VOLUME_1']
    )

    df['SV1_MISSING'] = (
        df['SIGNED_VOLUME_1'].isna().astype('int8')
    )


minimal_features += [
    'RET1_X_SV1',
    'SV1_MISSING'
]


# ============================================================
# COMMENTED OUT:
# ALLOCATION IDENTITY
# ============================================================

# minimal_features += ['ALLOCATION']

# X_train['ALLOCATION'] = (
#     X_train['ALLOCATION'].astype('category')
# )

# X_test['ALLOCATION'] = (
#     X_test['ALLOCATION'].astype('category')
# )


print("Minimal features:", minimal_features)

X_train_minimal = X_train.copy()

Train TS: 2522
Test TS: 120
Shared TS: 0
Test-only TS: 120
Train-only TS: 2522


/tmp/ipykernel_23368/4172087055.py:160: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[RET_features[:i]].mean(1)
/tmp/ipykernel_23368/4172087055.py:164: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_test[RET_features[:i]].mean(1)
/tmp/ipykernel_23368/4172087055.py:160: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[RET_features[:i]].mean(1)
/tmp/ipykernel_23368/4172087055.py:164: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_test[RET_features[:i]].mean(1)
/tmp/ipykernel_23368/4172087055.py:160: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_train[RET_features[:i]].mean(1)
/tmp/ipykernel_23368/4172087055.py:164: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  X_test[RET_features[:i

Minimal features: ['RET_1', 'RET_2', 'RET_3', 'RET_4', 'RET_5', 'RET_6', 'RET_7', 'RET_8', 'RET_9', 'RET_10', 'RET_11', 'RET_12', 'RET_13', 'RET_14', 'RET_15', 'RET_16', 'RET_17', 'RET_18', 'RET_19', 'RET_20', 'SIGNED_VOLUME_1', 'SIGNED_VOLUME_2', 'SIGNED_VOLUME_3', 'SIGNED_VOLUME_4', 'SIGNED_VOLUME_5', 'SIGNED_VOLUME_6', 'SIGNED_VOLUME_7', 'SIGNED_VOLUME_8', 'SIGNED_VOLUME_9', 'SIGNED_VOLUME_10', 'SIGNED_VOLUME_11', 'SIGNED_VOLUME_12', 'SIGNED_VOLUME_13', 'SIGNED_VOLUME_14', 'SIGNED_VOLUME_15', 'SIGNED_VOLUME_16', 'SIGNED_VOLUME_17', 'SIGNED_VOLUME_18', 'SIGNED_VOLUME_19', 'SIGNED_VOLUME_20', 'MEDIAN_DAILY_TURNOVER', 'AVERAGE_PERF_3', 'AVERAGE_PERF_5', 'AVERAGE_PERF_10', 'AVERAGE_PERF_15', 'AVERAGE_PERF_20', 'STD_PERF_20', 'ALLOCATIONS_STD_PERF_20', 'RET_1_ZSCORE', 'RET_2_ZSCORE', 'RET_3_ZSCORE', 'RET_4_ZSCORE', 'RET_5_ZSCORE', 'RET_6_ZSCORE', 'RET_7_ZSCORE', 'RET_8_ZSCORE', 'RET_9_ZSCORE', 'RET_10_ZSCORE', 'RET_11_ZSCORE', 'RET_12_ZSCORE', 'RET_13_ZSCORE', 'RET_14_ZSCORE', 'RET_15_ZS

/tmp/ipykernel_23368/4172087055.py:303: Pandas4Warning: Starting with pandas version 4.0 all arguments of std will be keyword-only.
  X_test[cols].std(1)


In [3]:
unique_ts = X_train_minimal["TS"].unique()

ts_tr, ts_val = train_test_split(
    unique_ts,
    test_size=0.20,
    random_state=42
)

train_mask = X_train_minimal["TS"].isin(ts_tr)
val_mask = X_train_minimal["TS"].isin(ts_val)

train_row_count = len(X_train_minimal)
model_inputs = pd.concat(
    [X_train_minimal[minimal_features], X_test[minimal_features]],
    axis=0,
)

categorical_features = [
    feature for feature in ["ALLOCATION"]
    if feature in model_inputs.columns
]
model_inputs = pd.get_dummies(
    model_inputs,
    columns=categorical_features,
    dtype=np.int8,
)
minimal_features = model_inputs.columns.tolist()

X_train_minimal = model_inputs.iloc[:train_row_count].copy()
X_test = model_inputs.iloc[train_row_count:].copy()
X_tr = X_train_minimal.loc[train_mask]
X_val = X_train_minimal.loc[val_mask]

y_tr = y_train_bin.loc[train_mask]
y_val = y_train_bin.loc[val_mask]

KeyError: "['ALLOCATIONS_STD_PERF_20'] not in index"

In [4]:

extra_model = ExtraTreesClassifier(
    n_estimators=10,
    max_features=0.2,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
extra_model.fit(X_tr[minimal_features], y_tr)

train_pred = extra_model.predict(X_tr[minimal_features])
val_pred = extra_model.predict(X_val[minimal_features])
train_accuracy = accuracy_score(y_tr, train_pred)
val_accuracy = accuracy_score(y_val, val_pred)
print("Extra Trees train accuracy:", train_accuracy)
print("Extra Trees validation accuracy:", val_accuracy)


NameError: name 'X_tr' is not defined

In [5]:
# ============================================================
# LOAD DATA
# ============================================================

X_train = pd.read_csv(
    '../data/X_train.csv',
    index_col='ROW_ID'
)

X_test = pd.read_csv(
    '../data/X_test.csv',
    index_col='ROW_ID'
)

y_train = pd.read_csv(
    '../data/y_train.csv',
    index_col='ROW_ID'
)

y_train_bin = (
    y_train['target'] > 0
).astype(int)

sample_submission = pd.read_csv(
    '../data/sample_submission.csv',
    index_col='ROW_ID'
)


# ============================================================
# CHECK TRAIN / TEST TS
# ============================================================

print("Train TS:", X_train['TS'].nunique())
print("Test TS:", X_test['TS'].nunique())

train_ts = set(X_train['TS'].unique())
test_ts = set(X_test['TS'].unique())

print("Shared TS:", len(train_ts & test_ts))
print("Test-only TS:", len(test_ts - train_ts))
print("Train-only TS:", len(train_ts - test_ts))


# ============================================================
# BASIC FEATURE GROUPS
# ============================================================

RET_features = [
    f'RET_{i}'
    for i in range(1, 21)
]

SIGNED_VOLUME_features = [
    f'SIGNED_VOLUME_{i}'
    for i in range(1, 21)
]

TURNOVER_features = [
    'MEDIAN_DAILY_TURNOVER'
]


# ============================================================
# NO CROSS-SECTIONAL Z-SCORES
#
# Deliberately removed for this experiment.
# ============================================================

# def add_cross_sectional_zscore(
#     X,
#     base_features,
#     suffix='_ZSCORE'
# ):
#     new_cols = []
#
#     for feat in base_features:
#
#         ts_mean = X.groupby('TS')[feat].transform('mean')
#         ts_std = X.groupby('TS')[feat].transform('std')
#
#         new_col = f'{feat}{suffix}'
#
#         X[new_col] = np.where(
#             ts_std > 0,
#             (X[feat] - ts_mean) / ts_std,
#             0.0
#         )
#
#         new_cols.append(new_col)
#
#     return X, new_cols
#
#
# RET_zscore_features = []


# ============================================================
# NO CROSS-SECTIONAL RET_1 MARKET AVERAGE
# ============================================================

# X_train['CROSS_SECTIONAL_AVG_RET1'] = (
#     X_train.groupby('TS')['RET_1'].transform('mean')
# )
#
# X_test['CROSS_SECTIONAL_AVG_RET1'] = (
#     X_test.groupby('TS')['RET_1'].transform('mean')
# )


# ============================================================
# INDIVIDUAL PERFORMANCE FEATURES
#
# KEEP
# ============================================================

for i in [3, 5, 10, 15, 20]:

    X_train[f'AVERAGE_PERF_{i}'] = (
        X_train[RET_features[:i]].mean(axis=1)
    )

    X_test[f'AVERAGE_PERF_{i}'] = (
        X_test[RET_features[:i]].mean(axis=1)
    )


# ============================================================
# INDIVIDUAL PERFORMANCE STD
#
# KEEP
# ============================================================

X_train['STD_PERF_20'] = (
    X_train[RET_features[:20]].std(axis=1)
)

X_test['STD_PERF_20'] = (
    X_test[RET_features[:20]].std(axis=1)
)


# ============================================================
# BASE INDIVIDUAL FEATURES
# ============================================================

features = (
    RET_features
    + SIGNED_VOLUME_features
    + TURNOVER_features
)

features += [
    f'AVERAGE_PERF_{i}'
    for i in [3, 5, 10, 15, 20]
]

features += [
    'STD_PERF_20'
]


# ============================================================
# NO CROSS-SECTIONAL PERFORMANCE FEATURES
#
# These are deliberately NOT added:
#
# ALLOCATIONS_AVERAGE_PERF_3
# ALLOCATIONS_AVERAGE_PERF_5
# ALLOCATIONS_AVERAGE_PERF_10
# ALLOCATIONS_AVERAGE_PERF_15
# ALLOCATIONS_AVERAGE_PERF_20
# ALLOCATIONS_STD_PERF_20
# ============================================================


# ============================================================
# INDIVIDUAL VOLUME FEATURES
#
# KEEP
# ============================================================

SIGNED_VOLUME_features_clean = [
    f'SIGNED_VOLUME_{i}'
    for i in range(2, 21)
]


for i in [3, 5, 10, 15, 20]:

    cols = SIGNED_VOLUME_features_clean[:i-1]

    X_train[f'AVERAGE_VOLUME_{i}'] = (
        X_train[cols].mean(axis=1)
    )

    X_test[f'AVERAGE_VOLUME_{i}'] = (
        X_test[cols].mean(axis=1)
    )


# ============================================================
# INDIVIDUAL VOLUME STD
#
# KEEP
# ============================================================

cols = SIGNED_VOLUME_features_clean[:19]

X_train['STD_VOLUME_20'] = (
    X_train[cols].std(axis=1)
)

X_test['STD_VOLUME_20'] = (
    X_test[cols].std(axis=1)
)


# ============================================================
# NO CROSS-SECTIONAL VOLUME FEATURES
#
# These are deliberately NOT created:
#
# ALLOCATIONS_AVERAGE_VOLUME_3
# ALLOCATIONS_AVERAGE_VOLUME_5
# ALLOCATIONS_AVERAGE_VOLUME_10
# ALLOCATIONS_AVERAGE_VOLUME_15
# ALLOCATIONS_AVERAGE_VOLUME_20
# ALLOCATIONS_STD_VOLUME_20
# ============================================================


# ============================================================
# ADD INDIVIDUAL VOLUME FEATURES
# ============================================================

volume_features = (
    [
        f'AVERAGE_VOLUME_{i}'
        for i in [3, 5, 10, 15, 20]
    ]
    + [
        'STD_VOLUME_20'
    ]
)

features += volume_features


# ============================================================
# INDIVIDUAL INTERACTION FEATURES
#
# KEEP
# ============================================================

for df in [X_train, X_test]:

    df['RET1_X_SV1'] = (
        df['RET_1']
        * df['SIGNED_VOLUME_1']
    )

    df['SV1_MISSING'] = (
        df['SIGNED_VOLUME_1']
        .isna()
        .astype('int8')
    )


# ============================================================
# FINAL FEATURE LIST
#
# IMPORTANT:
# No TS
# No ALLOCATION
# No cross-sectional z-scores
# No ALLOCATIONS_* features
# ============================================================

minimal_features = (
    features
    + [
        'RET1_X_SV1',
        'SV1_MISSING'
    ]
)


# Remove any accidental duplicates while preserving order
minimal_features = list(
    dict.fromkeys(minimal_features)
)


# ============================================================
# CHECK FEATURES
# ============================================================

print("\nNumber of features:", len(minimal_features))

print("\nFeatures:")
for feature in minimal_features:
    print(feature)


# ============================================================
# CREATE MODEL INPUTS
#
# Numeric-only model, so no ALLOCATION encoding is needed.
# ============================================================

X_train_minimal = X_train[minimal_features].copy()
X_test_minimal = X_test[minimal_features].copy()


# ============================================================
# CHECK TRAIN / TEST SHAPES
# ============================================================

print("\nTrain shape:", X_train_minimal.shape)
print("Test shape:", X_test_minimal.shape)


# ============================================================
# CHECK FOR MISSING VALUES
# ============================================================

print(
    "\nTrain missing values:",
    X_train_minimal.isna().sum().sum()
)

print(
    "Test missing values:",
    X_test_minimal.isna().sum().sum()
)

Train TS: 2522
Test TS: 120
Shared TS: 0
Test-only TS: 120
Train-only TS: 2522

Number of features: 55

Features:
RET_1
RET_2
RET_3
RET_4
RET_5
RET_6
RET_7
RET_8
RET_9
RET_10
RET_11
RET_12
RET_13
RET_14
RET_15
RET_16
RET_17
RET_18
RET_19
RET_20
SIGNED_VOLUME_1
SIGNED_VOLUME_2
SIGNED_VOLUME_3
SIGNED_VOLUME_4
SIGNED_VOLUME_5
SIGNED_VOLUME_6
SIGNED_VOLUME_7
SIGNED_VOLUME_8
SIGNED_VOLUME_9
SIGNED_VOLUME_10
SIGNED_VOLUME_11
SIGNED_VOLUME_12
SIGNED_VOLUME_13
SIGNED_VOLUME_14
SIGNED_VOLUME_15
SIGNED_VOLUME_16
SIGNED_VOLUME_17
SIGNED_VOLUME_18
SIGNED_VOLUME_19
SIGNED_VOLUME_20
MEDIAN_DAILY_TURNOVER
AVERAGE_PERF_3
AVERAGE_PERF_5
AVERAGE_PERF_10
AVERAGE_PERF_15
AVERAGE_PERF_20
STD_PERF_20
AVERAGE_VOLUME_3
AVERAGE_VOLUME_5
AVERAGE_VOLUME_10
AVERAGE_VOLUME_15
AVERAGE_VOLUME_20
STD_VOLUME_20
RET1_X_SV1
SV1_MISSING

Train shape: (527073, 55)
Test shape: (31870, 55)

Train missing values: 791533
Test missing values: 49715


In [6]:
# ============================================================
# TS-GROUPED SPLIT
# ============================================================

unique_ts = X_train['TS'].unique()

ts_tr, ts_val = train_test_split(
    unique_ts,
    test_size=0.20,
    random_state=42
)

train_mask = X_train['TS'].isin(ts_tr)
val_mask = X_train['TS'].isin(ts_val)


X_tr = X_train_minimal.loc[train_mask]
X_val = X_train_minimal.loc[val_mask]

y_tr = y_train_bin.loc[train_mask]
y_val = y_train_bin.loc[val_mask]


print("Training rows:", len(X_tr))
print("Validation rows:", len(X_val))
print("Training TS:", X_train.loc[train_mask, 'TS'].nunique())
print("Validation TS:", X_train.loc[val_mask, 'TS'].nunique())


# ============================================================
# EXTRA TREES
# ============================================================

model = ExtraTreesClassifier(
    n_estimators=10,
    max_features=0.2,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42
)


# ============================================================
# FIT
# ============================================================

model.fit(X_tr, y_tr)


# ============================================================
# VALIDATION
# ============================================================

train_pred = model.predict(X_tr)
val_pred = model.predict(X_val)

train_accuracy = accuracy_score(
    y_tr,
    train_pred
)

val_accuracy = accuracy_score(
    y_val,
    val_pred
)

print(
    "Extra Trees train accuracy:",
    train_accuracy
)

print(
    "Extra Trees validation accuracy:",
    val_accuracy
)

Training rows: 423749
Validation rows: 103324
Training TS: 2017
Validation TS: 505
Extra Trees train accuracy: 0.6552440241746883
Extra Trees validation accuracy: 0.5137238202160195


In [7]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score


# ============================================================
# LOAD DATA
# ============================================================

X_train = pd.read_csv(
    '../data/X_train.csv',
    index_col='ROW_ID'
)

X_test = pd.read_csv(
    '../data/X_test.csv',
    index_col='ROW_ID'
)

y_train = pd.read_csv(
    '../data/y_train.csv',
    index_col='ROW_ID'
)

y_train_bin = (
    y_train['target'] > 0
).astype(int)


# ============================================================
# BASE FEATURE GROUPS
# ============================================================

RET_features = [
    f'RET_{i}'
    for i in range(1, 21)
]

SIGNED_VOLUME_features = [
    f'SIGNED_VOLUME_{i}'
    for i in range(1, 21)
]


# ============================================================
# INDIVIDUAL HISTORY FEATURES
# ============================================================

for i in [3, 5, 10, 15, 20]:

    X_train[f'AVERAGE_PERF_{i}'] = (
        X_train[RET_features[:i]].mean(axis=1)
    )

    X_test[f'AVERAGE_PERF_{i}'] = (
        X_test[RET_features[:i]].mean(axis=1)
    )


X_train['STD_PERF_20'] = (
    X_train[RET_features].std(axis=1)
)

X_test['STD_PERF_20'] = (
    X_test[RET_features].std(axis=1)
)


# ============================================================
# VOLUME HISTORY FEATURES
# ============================================================

for i in [3, 5, 10, 15, 20]:

    cols = [
        f'SIGNED_VOLUME_{j}'
        for j in range(2, i + 1)
    ]

    X_train[f'AVERAGE_VOLUME_{i}'] = (
        X_train[cols].mean(axis=1)
    )

    X_test[f'AVERAGE_VOLUME_{i}'] = (
        X_test[cols].mean(axis=1)
    )


volume_std_cols = [
    f'SIGNED_VOLUME_{i}'
    for i in range(2, 21)
]

X_train['STD_VOLUME_20'] = (
    X_train[volume_std_cols].std(axis=1)
)

X_test['STD_VOLUME_20'] = (
    X_test[volume_std_cols].std(axis=1)
)


# ============================================================
# SIMPLE INDIVIDUAL INTERACTIONS
# ============================================================

for df in [X_train, X_test]:

    df['RET1_X_SV1'] = (
        df['RET_1'] *
        df['SIGNED_VOLUME_1']
    )

    df['SV1_MISSING'] = (
        df['SIGNED_VOLUME_1']
        .isna()
        .astype('int8')
    )


# ============================================================
# BASE INDIVIDUAL FEATURE LIST
# ============================================================

features = (
    RET_features
    + SIGNED_VOLUME_features
    + [
        'MEDIAN_DAILY_TURNOVER'
    ]
    + [
        f'AVERAGE_PERF_{i}'
        for i in [3, 5, 10, 15, 20]
    ]
    + [
        'STD_PERF_20'
    ]
    + [
        f'AVERAGE_VOLUME_{i}'
        for i in [3, 5, 10, 15, 20]
    ]
    + [
        'STD_VOLUME_20',
        'RET1_X_SV1',
        'SV1_MISSING'
    ]
)


# ============================================================
# CROSS-SECTIONAL FEATURES
#
# IMPORTANT:
# These use ONLY information available at the current TS.
#
# No target information is used.
# ============================================================

cross_sectional_features = []


# ------------------------------------------------------------
# 1. RET_i percentile/rank within TS
# ------------------------------------------------------------

for i in [1, 2, 3, 5, 10, 20]:

    col = f'RET_{i}'

    train_rank_col = f'{col}_CS_RANK'
    test_rank_col = f'{col}_CS_RANK'

    X_train[train_rank_col] = (
        X_train.groupby('TS')[col]
        .rank(method='average', pct=True)
    )

    X_test[test_rank_col] = (
        X_test.groupby('TS')[col]
        .rank(method='average', pct=True)
    )

    cross_sectional_features.append(
        train_rank_col
    )


# ------------------------------------------------------------
# 2. Signed-volume percentile/rank within TS
# ------------------------------------------------------------

for i in [1, 2, 3, 5, 10, 20]:

    col = f'SIGNED_VOLUME_{i}'

    rank_col = f'{col}_CS_RANK'

    X_train[rank_col] = (
        X_train.groupby('TS')[col]
        .rank(method='average', pct=True)
    )

    X_test[rank_col] = (
        X_test.groupby('TS')[col]
        .rank(method='average', pct=True)
    )

    cross_sectional_features.append(rank_col)


# ============================================================
# 3. DEVIATION FROM TS MEAN
# ============================================================

for i in [1, 2, 3, 5, 10, 20]:

    col = f'RET_{i}'

    mean_col = f'{col}_CS_MEAN'

    X_train[mean_col] = (
        X_train.groupby('TS')[col]
        .transform('mean')
    )

    X_test[mean_col] = (
        X_test.groupby('TS')[col]
        .transform('mean')
    )

    diff_col = f'{col}_VS_CS_MEAN'

    X_train[diff_col] = (
        X_train[col] -
        X_train[mean_col]
    )

    X_test[diff_col] = (
        X_test[col] -
        X_test[mean_col]
    )

    cross_sectional_features.append(diff_col)

    # We don't need the raw mean as a separate feature.
    X_train.drop(columns=[mean_col], inplace=True)
    X_test.drop(columns=[mean_col], inplace=True)


# ============================================================
# 4. DEVIATION FROM TS MEDIAN
# ============================================================

for i in [1, 2, 3, 5, 10, 20]:

    col = f'RET_{i}'

    median_col = f'{col}_CS_MEDIAN'

    X_train[median_col] = (
        X_train.groupby('TS')[col]
        .transform('median')
    )

    X_test[median_col] = (
        X_test.groupby('TS')[col]
        .transform('median')
    )

    diff_col = f'{col}_VS_CS_MEDIAN'

    X_train[diff_col] = (
        X_train[col] -
        X_train[median_col]
    )

    X_test[diff_col] = (
        X_test[col] -
        X_test[median_col]
    )

    cross_sectional_features.append(diff_col)

    X_train.drop(columns=[median_col], inplace=True)
    X_test.drop(columns=[median_col], inplace=True)


# ============================================================
# 5. CROSS-SECTIONAL Z-SCORE
#
# Add only for selected recent periods.
# ============================================================

for i in [1, 2, 3, 5, 10, 20]:

    col = f'RET_{i}'

    cs_mean = (
        X_train.groupby('TS')[col]
        .transform('mean')
    )

    cs_std = (
        X_train.groupby('TS')[col]
        .transform('std')
    )

    z_col = f'{col}_CS_Z'

    X_train[z_col] = np.where(
        cs_std > 0,
        (X_train[col] - cs_mean) / cs_std,
        0.0
    )


    # Test must calculate its own snapshot statistics.
    test_mean = (
        X_test.groupby('TS')[col]
        .transform('mean')
    )

    test_std = (
        X_test.groupby('TS')[col]
        .transform('std')
    )

    X_test[z_col] = np.where(
        test_std > 0,
        (X_test[col] - test_mean) / test_std,
        0.0
    )

    cross_sectional_features.append(z_col)


# ============================================================
# 6. RELATIVE RECENT PERFORMANCE
# ============================================================

X_train['RET1_MINUS_RET2'] = (
    X_train['RET_1'] -
    X_train['RET_2']
)

X_test['RET1_MINUS_RET2'] = (
    X_test['RET_1'] -
    X_test['RET_2']
)


X_train['RET1_MINUS_RET5'] = (
    X_train['RET_1'] -
    X_train['RET_5']
)

X_test['RET1_MINUS_RET5'] = (
    X_test['RET_1'] -
    X_test['RET_5']
)


X_train['RET1_MINUS_RET20'] = (
    X_train['RET_1'] -
    X_train['RET_20']
)

X_test['RET1_MINUS_RET20'] = (
    X_test['RET_1'] -
    X_test['RET_20']
)


features += [
    'RET1_MINUS_RET2',
    'RET1_MINUS_RET5',
    'RET1_MINUS_RET20'
]


# ============================================================
# ADD CROSS-SECTIONAL FEATURES
# ============================================================

features += cross_sectional_features


# Remove duplicates
features = list(
    dict.fromkeys(features)
)


# ============================================================
# IMPORTANT CHECK
# ============================================================

print(
    "\nNumber of total features:",
    len(features)
)

print(
    "Cross-sectional features:",
    len(cross_sectional_features)
)

print("\nCross-sectional features:")
for f in cross_sectional_features:
    print(f)


# ============================================================
# CREATE MODEL DATA
# ============================================================

X_train_model = X_train[features].copy()
X_test_model = X_test[features].copy()


# ============================================================
# CHECK MISSING VALUES
# ============================================================

print(
    "\nTrain missing values:",
    X_train_model.isna().sum().sum()
)

print(
    "Test missing values:",
    X_test_model.isna().sum().sum()
)


# ============================================================
# TS-GROUPED TRAIN / VALIDATION SPLIT
# ============================================================

unique_ts = X_train['TS'].unique()

ts_tr, ts_val = train_test_split(
    unique_ts,
    test_size=0.20,
    random_state=42
)

train_mask = X_train['TS'].isin(ts_tr)
val_mask = X_train['TS'].isin(ts_val)


X_tr = X_train_model.loc[train_mask]
X_val = X_train_model.loc[val_mask]

y_tr = y_train_bin.loc[train_mask]
y_val = y_train_bin.loc[val_mask]


print("\nTraining rows:", len(X_tr))
print("Validation rows:", len(X_val))

print(
    "Training TS:",
    X_train.loc[train_mask, 'TS'].nunique()
)

print(
    "Validation TS:",
    X_train.loc[val_mask, 'TS'].nunique()
)


# ============================================================
# EXTRA TREES
#
# SAME CONFIG AS BEFORE
# ============================================================

model = ExtraTreesClassifier(
    n_estimators=10,
    max_features=0.2,
    min_samples_leaf=50,
    n_jobs=-1,
    random_state=42
)


# ============================================================
# TRAIN
# ============================================================

model.fit(
    X_tr,
    y_tr
)


# ============================================================
# PREDICT
# ============================================================

train_pred = model.predict(X_tr)
val_pred = model.predict(X_val)


# ============================================================
# ACCURACY
# ============================================================

train_accuracy = accuracy_score(
    y_tr,
    train_pred
)

val_accuracy = accuracy_score(
    y_val,
    val_pred
)


print(
    "\nExtra Trees train accuracy:",
    train_accuracy
)

print(
    "Extra Trees TS-grouped validation accuracy:",
    val_accuracy
)


# ============================================================
# FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
})

importance = (
    importance
    .sort_values(
        'importance',
        ascending=False
    )
    .reset_index(drop=True)
)


print("\nTop 30 features:")
print(
    importance.head(30).to_string(index=False)
)


Number of total features: 88
Cross-sectional features: 30

Cross-sectional features:
RET_1_CS_RANK
RET_2_CS_RANK
RET_3_CS_RANK
RET_5_CS_RANK
RET_10_CS_RANK
RET_20_CS_RANK
SIGNED_VOLUME_1_CS_RANK
SIGNED_VOLUME_2_CS_RANK
SIGNED_VOLUME_3_CS_RANK
SIGNED_VOLUME_5_CS_RANK
SIGNED_VOLUME_10_CS_RANK
SIGNED_VOLUME_20_CS_RANK
RET_1_VS_CS_MEAN
RET_2_VS_CS_MEAN
RET_3_VS_CS_MEAN
RET_5_VS_CS_MEAN
RET_10_VS_CS_MEAN
RET_20_VS_CS_MEAN
RET_1_VS_CS_MEDIAN
RET_2_VS_CS_MEDIAN
RET_3_VS_CS_MEDIAN
RET_5_VS_CS_MEDIAN
RET_10_VS_CS_MEDIAN
RET_20_VS_CS_MEDIAN
RET_1_CS_Z
RET_2_CS_Z
RET_3_CS_Z
RET_5_CS_Z
RET_10_CS_Z
RET_20_CS_Z

Train missing values: 1187844
Test missing values: 74616

Training rows: 423749
Validation rows: 103324
Training TS: 2017
Validation TS: 505

Extra Trees train accuracy: 0.6870387894720695
Extra Trees TS-grouped validation accuracy: 0.5192694824048624

Top 30 features:
                 feature  importance
           RET_1_CS_RANK    0.040829
          RET_20_CS_RANK    0.023904
          RE

In [8]:
ts_stats = (
    X_train
    .groupby('TS')
    .agg(
        N_ALLOCATIONS=('RET_1', 'size'),

        RET1_MEAN=('RET_1', 'mean'),
        RET1_STD=('RET_1', 'std'),
        RET1_MEDIAN=('RET_1', 'median'),

        RET2_MEAN=('RET_2', 'mean'),
        RET2_STD=('RET_2', 'std'),

        RET5_MEAN=('RET_5', 'mean'),
        RET5_STD=('RET_5', 'std'),

        RET20_MEAN=('RET_20', 'mean'),
        RET20_STD=('RET_20', 'std'),

        SV1_MEAN=('SIGNED_VOLUME_1', 'mean'),
        SV1_STD=('SIGNED_VOLUME_1', 'std'),

        TURNOVER_MEAN=('MEDIAN_DAILY_TURNOVER', 'mean'),
        TURNOVER_STD=('MEDIAN_DAILY_TURNOVER', 'std'),
    )
)

# Percentage of allocations with positive current return
ts_stats['RET1_POSITIVE_RATE'] = (
    X_train
    .groupby('TS')['RET_1']
    .apply(lambda x: (x > 0).mean())
)

# Target positive rate
ts_stats['TARGET_POSITIVE_RATE'] = (
    y_train_bin
    .groupby(X_train['TS'])
    .mean()
)
print(ts_stats.describe())

       N_ALLOCATIONS    RET1_MEAN     RET1_STD  RET1_MEDIAN    RET2_MEAN  \
count    2522.000000  2522.000000  2522.000000  2522.000000  2522.000000   
mean      208.990087     0.000080     0.002860     0.000078     0.000044   
std        81.903327     0.000852     0.000985     0.000682     0.000875   
min        65.000000    -0.005950     0.001024    -0.005588    -0.007597   
25%       140.000000    -0.000372     0.002197    -0.000269    -0.000421   
50%       276.000000     0.000080     0.002652     0.000074     0.000038   
75%       276.000000     0.000495     0.003248     0.000414     0.000491   
max       276.000000     0.005908     0.011833     0.003732     0.005871   

          RET2_STD    RET5_MEAN     RET5_STD   RET20_MEAN    RET20_STD  \
count  2522.000000  2522.000000  2522.000000  2522.000000  2522.000000   
mean      0.002903     0.000034     0.002886     0.000020     0.002938   
std       0.000950     0.000864     0.000906     0.000923     0.000903   
min       0.001147 

In [9]:
correct = (
    val_pred == y_val.values
)

val_results = pd.DataFrame({
    'TS': X_train.loc[val_mask, 'TS'].values,
    'correct': correct
})

ts_accuracy = (
    val_results
    .groupby('TS')['correct']
    .agg(
        accuracy='mean',
        n='size'
    )
)

In [10]:
ts_analysis = ts_stats.join(
    ts_accuracy,
    how='inner'
)

In [11]:
ts_analysis[
    ['RET1_STD', 'accuracy']
].corr()
ts_analysis[
    ['RET1_MEAN', 'accuracy']
].corr()
ts_analysis[
    ['N_ALLOCATIONS', 'accuracy']
].corr()
ts_analysis[
    ['SV1_STD', 'accuracy']
].corr()
ts_analysis['RET1_STD_BIN'] = pd.qcut(
    ts_analysis['RET1_STD'],
    10,
    duplicates='drop'
)

print(
    ts_analysis
    .groupby('RET1_STD_BIN', observed=True)['accuracy']
    .agg(['mean', 'count'])
)

                                       mean  count
RET1_STD_BIN                                      
(0.00023999999999999998, 0.00185]  0.511820     51
(0.00185, 0.00209]                 0.513114     50
(0.00209, 0.00224]                 0.499643     51
(0.00224, 0.00244]                 0.525056     50
(0.00244, 0.00261]                 0.521686     51
(0.00261, 0.00282]                 0.533343     50
(0.00282, 0.00311]                 0.522262     50
(0.00311, 0.00348]                 0.529345     51
(0.00348, 0.0042]                  0.535207     50
(0.0042, 0.00983]                  0.544341     51


In [12]:
train_ts_ret1_std = (
    X_train
    .groupby('TS')['RET_1']
    .std()
)

test_ts_ret1_std = (
    X_test
    .groupby('TS')['RET_1']
    .std()
)

print("TRAIN")
print(train_ts_ret1_std.describe())

print("\nTEST")
print(test_ts_ret1_std.describe())

TRAIN
count    2522.000000
mean        0.002860
std         0.000985
min         0.001024
25%         0.002197
50%         0.002652
75%         0.003248
max         0.011833
Name: RET_1, dtype: float64

TEST
count    120.000000
mean       0.002724
std        0.000959
min        0.001411
25%        0.002128
50%        0.002600
75%        0.002998
max        0.008266
Name: RET_1, dtype: float64


In [13]:
candidate_snapshot_features = [
    'N_ALLOCATIONS',
    'RET1_MEAN',
    'RET1_STD',
    'RET1_MEDIAN',
    'RET2_MEAN',
    'RET2_STD',
    'RET5_MEAN',
    'RET5_STD',
    'RET20_MEAN',
    'RET20_STD',
    'SV1_MEAN',
    'SV1_STD',
    'TURNOVER_MEAN',
    'TURNOVER_STD',
    'RET1_POSITIVE_RATE',
    'TARGET_POSITIVE_RATE'
]

corrs = (
    ts_analysis[candidate_snapshot_features + ['accuracy']]
    .corr()['accuracy']
    .drop('accuracy')
    .sort_values()
)

print(corrs)

N_ALLOCATIONS          -0.181099
TURNOVER_STD           -0.057006
TURNOVER_MEAN          -0.046362
SV1_MEAN               -0.029659
RET1_MEDIAN            -0.026923
RET5_STD               -0.011263
RET20_MEAN             -0.007290
RET1_MEAN              -0.006558
RET1_POSITIVE_RATE      0.001619
RET5_MEAN               0.013817
RET20_STD               0.024510
SV1_STD                 0.032537
RET2_STD                0.080161
RET2_MEAN               0.085190
RET1_STD                0.131647
TARGET_POSITIVE_RATE    0.208141
Name: accuracy, dtype: float64


In [14]:
ts_analysis['N_ALLOC_BIN'] = pd.qcut(
    ts_analysis['N_ALLOCATIONS'],
    q=10,
    duplicates='drop'
)

print(
    ts_analysis.groupby('N_ALLOC_BIN', observed=True)
    .agg(
        accuracy=('accuracy', 'mean'),
        n_ts=('accuracy', 'size'),
        avg_allocations=('N_ALLOCATIONS', 'mean')
    )
)

                accuracy  n_ts  avg_allocations
N_ALLOC_BIN                                    
(64.999, 69.0]  0.538339   105        66.666667
(69.0, 204.0]   0.541668    73       166.000000
(204.0, 206.0]  0.517693    76       206.000000
(206.0, 276.0]  0.513887   251       273.107570


In [15]:
ts_analysis['FULL_SNAPSHOT'] = (
    ts_analysis['N_ALLOCATIONS'] == 276
)

print(
    ts_analysis.groupby('FULL_SNAPSHOT')
    .agg(
        accuracy=('accuracy', 'mean'),
        n_ts=('accuracy', 'size'),
        avg_allocations=('N_ALLOCATIONS', 'mean')
    )
)

               accuracy  n_ts  avg_allocations
FULL_SNAPSHOT                                 
False          0.532660   265       139.939623
True           0.513511   240       276.000000


In [16]:
# -----------------------------------------
# EXPERIMENT 2: Validation vs Test
# snapshot size distribution
# -----------------------------------------

# Validation snapshots
val_n_alloc = (
    X_train.loc[val_mask]
    .groupby('TS')
    .size()
    .rename('N_ALLOCATIONS')
)

# Test snapshots
test_n_alloc = (
    X_test
    .groupby('TS')
    .size()
    .rename('N_ALLOCATIONS')
)

print("===== VALIDATION =====")
print(val_n_alloc.describe())

print("\n===== TEST =====")
print(test_n_alloc.describe())


# -----------------------------------------
# Full snapshot proportion
# -----------------------------------------

val_full_pct = (val_n_alloc == 276).mean()
test_full_pct = (test_n_alloc == 276).mean()

print("\n===== FULL SNAPSHOTS =====")
print(f"Validation: {val_full_pct:.3%}")
print(f"Test:       {test_full_pct:.3%}")


# -----------------------------------------
# Exact distribution of snapshot sizes
# -----------------------------------------

print("\n===== VALIDATION COUNTS =====")
print(val_n_alloc.value_counts().sort_index())

print("\n===== TEST COUNTS =====")
print(test_n_alloc.value_counts().sort_index())

===== VALIDATION =====
count    505.000000
mean     204.601980
std       82.779673
min       65.000000
25%      138.000000
50%      206.000000
75%      276.000000
max      276.000000
Name: N_ALLOCATIONS, dtype: float64

===== TEST =====
count    120.000000
mean     265.583333
std       32.054466
min       70.000000
25%      277.500000
50%      278.000000
75%      278.000000
max      278.000000
Name: N_ALLOCATIONS, dtype: float64

===== FULL SNAPSHOTS =====
Validation: 47.525%
Test:       10.000%

===== VALIDATION COUNTS =====
N_ALLOCATIONS
65      61
68       1
69      43
70       5
135      2
136      7
138      9
140     14
204     36
206     76
210     11
276    240
Name: count, dtype: int64

===== TEST COUNTS =====
N_ALLOCATIONS
70      1
140     1
206     2
208    12
210     2
276    12
278    90
Name: count, dtype: int64


In [17]:
bins = [-np.inf, 100, 200, 210, 275, 276]
labels = [
    '<=100',
    '101-200',
    '201-210',
    '211-275',
    '276'
]

val_bins = pd.cut(
    val_n_alloc,
    bins=bins,
    labels=labels
)

test_bins = pd.cut(
    test_n_alloc,
    bins=bins,
    labels=labels
)

comparison = pd.DataFrame({
    'validation_count': val_bins.value_counts().sort_index(),
    'test_count': test_bins.value_counts().sort_index()
})

comparison['validation_pct'] = (
    comparison['validation_count'] / len(val_n_alloc) * 100
)

comparison['test_pct'] = (
    comparison['test_count'] / len(test_n_alloc) * 100
)

print(comparison)

               validation_count  test_count  validation_pct   test_pct
N_ALLOCATIONS                                                         
<=100                       110           1       21.782178   0.833333
101-200                      32           1        6.336634   0.833333
201-210                     123          16       24.356436  13.333333
211-275                       0           0        0.000000   0.000000
276                         240          12       47.524752  10.000000


In [18]:
# Allocations in training
train_allocations = set(X_train['ALLOCATION'].unique())

# Allocations in test
test_allocations = set(X_test['ALLOCATION'].unique())

print("Unique train allocations:", len(train_allocations))
print("Unique test allocations:", len(test_allocations))

print("\nTest allocations NOT seen in train:")
print(sorted(test_allocations - train_allocations))

print("\nTrain allocations NOT seen in test:")
print(sorted(train_allocations - test_allocations))

Unique train allocations: 278
Unique test allocations: 278

Test allocations NOT seen in train:
[]

Train allocations NOT seen in test:
[]


In [19]:
test_278_ts = (
    X_test.groupby('TS')
    .size()
)

test_278_ts = test_278_ts[test_278_ts == 278].index

test_278 = X_test[X_test['TS'].isin(test_278_ts)]

print("278-allocation test snapshots:", len(test_278_ts))
print("Rows:", len(test_278))

print("\nUnique allocations in 278 snapshots:")
print(test_278['ALLOCATION'].nunique())

278-allocation test snapshots: 90
Rows: 25020

Unique allocations in 278 snapshots:
278


In [20]:
alloc_counts_278 = (
    test_278.groupby('ALLOCATION')['TS']
    .nunique()
    .sort_values()
)

print(alloc_counts_278.head(20))
print(alloc_counts_278.tail(20))

ALLOCATION
ALLOCATION_01     90
ALLOCATION_02     90
ALLOCATION_03     90
ALLOCATION_04     90
ALLOCATION_05     90
ALLOCATION_06     90
ALLOCATION_07     90
ALLOCATION_08     90
ALLOCATION_09     90
ALLOCATION_10     90
ALLOCATION_100    90
ALLOCATION_101    90
ALLOCATION_102    90
ALLOCATION_103    90
ALLOCATION_104    90
ALLOCATION_105    90
ALLOCATION_106    90
ALLOCATION_107    90
ALLOCATION_108    90
ALLOCATION_109    90
Name: TS, dtype: int64
ALLOCATION
ALLOCATION_80    90
ALLOCATION_81    90
ALLOCATION_82    90
ALLOCATION_83    90
ALLOCATION_84    90
ALLOCATION_85    90
ALLOCATION_86    90
ALLOCATION_87    90
ALLOCATION_88    90
ALLOCATION_89    90
ALLOCATION_90    90
ALLOCATION_91    90
ALLOCATION_92    90
ALLOCATION_93    90
ALLOCATION_94    90
ALLOCATION_95    90
ALLOCATION_96    90
ALLOCATION_97    90
ALLOCATION_98    90
ALLOCATION_99    90
Name: TS, dtype: int64


In [21]:
def snapshot_signature(df):
    return (
        df.groupby('TS')['ALLOCATION']
        .apply(lambda x: tuple(sorted(x)))
        .rename('signature')
    )

train_signatures = snapshot_signature(X_train)
test_signatures = snapshot_signature(X_test)

print("Unique train snapshot configurations:",
      train_signatures.nunique())

print("Unique test snapshot configurations:",
      test_signatures.nunique())
print("\nTRAIN CONFIGURATION COUNTS")
print(train_signatures.value_counts().head(20))

print("\nTEST CONFIGURATION COUNTS")
print(test_signatures.value_counts().head(20))

Unique train snapshot configurations: 24
Unique test snapshot configurations: 10

TRAIN CONFIGURATION COUNTS
signature
(ALLOCATION_01, ALLOCATION_02, ALLOCATION_03, ALLOCATION_04, ALLOCATION_05, ALLOCATION_06, ALLOCATION_07, ALLOCATION_08, ALLOCATION_09, ALLOCATION_10, ALLOCATION_100, ALLOCATION_101, ALLOCATION_102, ALLOCATION_103, ALLOCATION_104, ALLOCATION_105, ALLOCATION_106, ALLOCATION_107, ALLOCATION_108, ALLOCATION_109, ALLOCATION_11, ALLOCATION_110, ALLOCATION_111, ALLOCATION_112, ALLOCATION_113, ALLOCATION_114, ALLOCATION_115, ALLOCATION_116, ALLOCATION_117, ALLOCATION_118, ALLOCATION_119, ALLOCATION_12, ALLOCATION_120, ALLOCATION_121, ALLOCATION_122, ALLOCATION_123, ALLOCATION_124, ALLOCATION_125, ALLOCATION_126, ALLOCATION_127, ALLOCATION_128, ALLOCATION_129, ALLOCATION_13, ALLOCATION_130, ALLOCATION_131, ALLOCATION_132, ALLOCATION_133, ALLOCATION_134, ALLOCATION_135, ALLOCATION_136, ALLOCATION_137, ALLOCATION_138, ALLOCATION_139, ALLOCATION_140, ALLOCATION_141, ALLOCATION_14

In [22]:
# Pick one 276 snapshot
ts_276 = test_n_alloc[test_n_alloc == 276].index[0]

# Pick one 278 snapshot
ts_278 = test_n_alloc[test_n_alloc == 278].index[0]

alloc_276 = set(
    X_test.loc[X_test['TS'] == ts_276, 'ALLOCATION']
)

alloc_278 = set(
    X_test.loc[X_test['TS'] == ts_278, 'ALLOCATION']
)

print("276 snapshot missing:")
print(sorted(set(X_test['ALLOCATION'].unique()) - alloc_276))

print("\n278 snapshot missing:")
print(sorted(set(X_test['ALLOCATION'].unique()) - alloc_278))

276 snapshot missing:
['ALLOCATION_14', 'ALLOCATION_46']

278 snapshot missing:
[]


In [23]:
all_allocations = sorted(X_train['ALLOCATION'].unique())

presence = (
    X_train
    .assign(present=1)
    .pivot_table(
        index='TS',
        columns='ALLOCATION',
        values='present',
        fill_value=0
    )
)

print(presence.shape)

(2522, 278)


In [24]:
# For every allocation, how many training snapshots contain it?
allocation_presence = (
    presence.mean()
    .sort_values()
)

print(allocation_presence.head(30))
print("\n--- highest ---")
print(allocation_presence.tail(30))

ALLOCATION
ALLOCATION_46     0.007534
ALLOCATION_14     0.007534
ALLOCATION_01     0.567407
ALLOCATION_92     0.567407
ALLOCATION_91     0.567407
ALLOCATION_102    0.567407
ALLOCATION_100    0.567407
ALLOCATION_10     0.567407
ALLOCATION_03     0.567407
ALLOCATION_06     0.567407
ALLOCATION_94     0.567407
ALLOCATION_13     0.567407
ALLOCATION_131    0.567407
ALLOCATION_117    0.567407
ALLOCATION_67     0.567407
ALLOCATION_66     0.567407
ALLOCATION_71     0.567407
ALLOCATION_72     0.567407
ALLOCATION_119    0.567407
ALLOCATION_163    0.567407
ALLOCATION_161    0.567407
ALLOCATION_154    0.567407
ALLOCATION_178    0.567407
ALLOCATION_140    0.567407
ALLOCATION_175    0.567407
ALLOCATION_177    0.567407
ALLOCATION_136    0.567407
ALLOCATION_122    0.567407
ALLOCATION_141    0.567407
ALLOCATION_167    0.567407
dtype: float64

--- highest ---
ALLOCATION
ALLOCATION_203    0.96709
ALLOCATION_204    0.96709
ALLOCATION_182    0.96709
ALLOCATION_183    0.96709
ALLOCATION_191    0.96709
ALLOCA

In [ ]:
train_n_alloc = X_train.groupby('TS').size()

print("Training snapshots with 278:")
print((train_n_alloc == 278).sum())

print("Training snapshots with 277:")
print((train_n_alloc == 277).sum())

print("Training max:")
print(train_n_alloc.max())

Training snapshots with 278:
0
Training snapshots with 277:
0
Training max:
276


In [25]:
def config_id(df):
    return (
        df.groupby('TS')['ALLOCATION']
        .apply(lambda x: tuple(sorted(x)))
    )

train_config = config_id(X_train)
test_config = config_id(X_test)

# Map each full configuration to a compact integer label.
all_configs = pd.concat([train_config, test_config])
config_labels = pd.factorize(all_configs)[0]

train_labels = pd.Series(
    config_labels[:len(train_config)],
    index=train_config.index,
    name='CONFIG_ID'
)
test_labels = pd.Series(
    config_labels[len(train_config):],
    index=test_config.index,
    name='CONFIG_ID'
)

def summarize_configs(config_labels, df):
    return (
        pd.DataFrame({
            'CONFIG_ID': config_labels,
            'N_ALLOCATIONS': df.groupby('TS').size().values
        })
        .value_counts()
        .rename('N_SNAPSHOTS')
        .reset_index()
        .sort_values('N_SNAPSHOTS', ascending=False)
    )

print('TRAIN CONFIGURATIONS')
print(summarize_configs(train_labels, X_train).head(30).to_string(index=False))

print('\nTEST CONFIGURATIONS')
print(summarize_configs(test_labels, X_test).head(30).to_string(index=False))

TRAIN CONFIGURATIONS
 CONFIG_ID  N_ALLOCATIONS  N_SNAPSHOTS
         0            276         1273
         4            206          304
         3             65          265
         1             69          213
         2            204          173
        13            210           47
         9            138           44
         8            206           28
        10            206           26
        11            140           22
         5            206           19
         6             68           18
        17            140           17
        12             70           15
        15            136           11
        19            140           10
        21            136           10
        22            135            7
        18            136            5
        14            135            4
        16             70            4
        20             66            4
         7             69            2
        23             70            1

TES

In [26]:
def snapshot_stats(df):
    return df.groupby('TS').agg(
        N_ALLOCATIONS=('ALLOCATION', 'size'),
        RET1_MEAN=('RET_1', 'mean'),
        RET1_STD=('RET_1', 'std'),
        RET1_POSITIVE_RATE=('RET_1', lambda x: (x > 0).mean()),
        RET2_MEAN=('RET_2', 'mean'),
        RET2_STD=('RET_2', 'std'),
        SV1_MEAN=('SIGNED_VOLUME_1', 'mean'),
        SV1_STD=('SIGNED_VOLUME_1', 'std'),
    )

train_stats = snapshot_stats(X_train)
test_stats = snapshot_stats(X_test)

train_276 = train_stats[train_stats['N_ALLOCATIONS'] == 276]
test_278 = test_stats[test_stats['N_ALLOCATIONS'] == 278]

print("TRAIN 276")
print(train_276.describe().T)

print("\nTEST 278")
print(test_278.describe().T)

TRAIN 276
                     count        mean       std         min         25%  \
N_ALLOCATIONS       1273.0  276.000000  0.000000  276.000000  276.000000   
RET1_MEAN           1273.0    0.000059  0.000696   -0.002834   -0.000354   
RET1_STD            1273.0    0.002635  0.000809    0.001393    0.002104   
RET1_POSITIVE_RATE  1273.0    0.511410  0.083981    0.246377    0.449275   
RET2_MEAN           1273.0    0.000027  0.000710   -0.002575   -0.000414   
RET2_STD            1273.0    0.002696  0.000784    0.001307    0.002158   
SV1_MEAN            1273.0    0.151693  0.300293   -1.327751   -0.023024   
SV1_STD             1273.0    1.488026  0.704695    0.466595    1.036406   

                           50%         75%         max  
N_ALLOCATIONS       276.000000  276.000000  276.000000  
RET1_MEAN             0.000057    0.000444    0.005192  
RET1_STD              0.002469    0.002958    0.009829  
RET1_POSITIVE_RATE    0.514493    0.572464    0.768116  
RET2_MEAN           

In [27]:
train_276_ts = X_train.loc[
    X_train.groupby('TS')['ALLOCATION'].transform('size') == 276,
    ['TS', 'ALLOCATION']
]

test_278_ts = X_test.loc[
    X_test.groupby('TS')['ALLOCATION'].transform('size') == 278,
    ['TS', 'ALLOCATION']
]

train_allocs_276 = set(train_276_ts['ALLOCATION'].unique())
test_allocs_278 = set(test_278_ts['ALLOCATION'].unique())

print("In 278 test but not 276 train:")
print(sorted(test_allocs_278 - train_allocs_276))

print("\nIn 276 train but not 278 test:")
print(sorted(train_allocs_276 - test_allocs_278))

In 278 test but not 276 train:
['ALLOCATION_14', 'ALLOCATION_46']

In 276 train but not 278 test:
[]


In [28]:
for alloc in ['ALLOCATION_14', 'ALLOCATION_46']:
    s = X_train[X_train['ALLOCATION'] == alloc]

    print(
        alloc,
        "n =", len(s),
        "target positive rate =", y_train.loc[s.index].mean(),
        "RET1 mean =", s['RET_1'].mean(),
        "RET1 std =", s['RET_1'].std(),
    )

print("\nAll other allocations:")
mask = ~X_train['ALLOCATION'].isin(['ALLOCATION_14', 'ALLOCATION_46'])

print(
    "target positive rate =", y_train.loc[X_train.index[mask]].mean(),
    "RET1 mean =", X_train.loc[mask, 'RET_1'].mean(),
    "RET1 std =", X_train.loc[mask, 'RET_1'].std(),
)

ALLOCATION_14 n = 19 target positive rate = target   -0.004346
dtype: float64 RET1 mean = -0.005561391621052631 RET1 std = 0.013357383518699336
ALLOCATION_46 n = 19 target positive rate = target   -0.004346
dtype: float64 RET1 mean = -0.005561391621052631 RET1 std = 0.013357383518699336

All other allocations:
target positive rate = target    0.000035
dtype: float64 RET1 mean = 7.899112080478702e-05 RET1 std = 0.0030388198196811197


In [29]:
print(
    X_train[X_train['ALLOCATION'] == 'ALLOCATION_14']
    [['TS', 'ALLOCATION', 'RET_1', 'RET_2', 'RET_3', 'RET_4', 'RET_5']]
    .sort_values('TS')
    .to_string(index=False)
)


       TS    ALLOCATION     RET_1     RET_2     RET_3     RET_4     RET_5
DATE_0013 ALLOCATION_14 -0.007830 -0.018833 -0.007667  0.001738 -0.000517
DATE_0156 ALLOCATION_14 -0.003300 -0.032963 -0.016905  0.011786  0.002345
DATE_0395 ALLOCATION_14 -0.005926 -0.000885 -0.033068 -0.017079  0.012081
DATE_0399 ALLOCATION_14 -0.016199 -0.007763  0.001890 -0.000650  0.017475
DATE_0489 ALLOCATION_14 -0.033911  0.006756  0.007001  0.008951 -0.003861
DATE_0835 ALLOCATION_14 -0.012253 -0.043723  0.007405  0.007963  0.007940
DATE_0878 ALLOCATION_14 -0.016216 -0.015010 -0.043676  0.007744  0.007701
DATE_0945 ALLOCATION_14  0.013178  0.013557 -0.018263 -0.015601 -0.043621
DATE_1095 ALLOCATION_14  0.010345 -0.002740  0.001502 -0.000577 -0.001294
DATE_1618 ALLOCATION_14 -0.003026  0.007337  0.008846 -0.003630  0.001224
DATE_1734 ALLOCATION_14  0.006287  0.009588 -0.003111  0.001711 -0.000570
DATE_1823 ALLOCATION_14 -0.012007  0.013013  0.003438  0.016267 -0.005700
DATE_1861 ALLOCATION_14 -0.004962  0.0

In [30]:
a14 = X_train[X_train['ALLOCATION'] == 'ALLOCATION_14'].sort_values('TS').reset_index(drop=True)
a46 = X_train[X_train['ALLOCATION'] == 'ALLOCATION_46'].sort_values('TS').reset_index(drop=True)

print("Same TS:", (a14['TS'].values == a46['TS'].values).all())

feature_cols = [
    c for c in X_train.columns
    if c not in ['TS', 'ALLOCATION']
]

diffs = []
for c in feature_cols:
    if not a14[c].equals(a46[c]):
        diffs.append(c)

print("Different feature columns:", diffs)

Same TS: True
Different feature columns: ['MEDIAN_DAILY_TURNOVER']


In [31]:
from sklearn.metrics import accuracy_score

val_df = X_train.loc[val_mask].copy()
val_y = y_train_bin.loc[val_mask].squeeze()

for i in range(1, 21):
    pred = (val_df[f'RET_{i}'] > 0).astype(int)
    acc = accuracy_score(val_y, pred)
    print(f"RET_{i:2d}: {acc:.4f}")
for i in range(1, 21):
    x = val_df[f'RET_{i}']
    pred_same = (x > 0).astype(int)
    pred_flip = (x < 0).astype(int)

    a_same = accuracy_score(val_y, pred_same)
    a_flip = accuracy_score(val_y, pred_flip)

    print(
        f"RET_{i:2d}: same={a_same:.4f}, flip={a_flip:.4f}"
    )

RET_ 1: 0.5216
RET_ 2: 0.4975
RET_ 3: 0.4968
RET_ 4: 0.4982
RET_ 5: 0.4967
RET_ 6: 0.5037
RET_ 7: 0.5080
RET_ 8: 0.5003
RET_ 9: 0.5061
RET_10: 0.5048
RET_11: 0.5045
RET_12: 0.5064
RET_13: 0.5004
RET_14: 0.4948
RET_15: 0.4971
RET_16: 0.4999
RET_17: 0.5045
RET_18: 0.4996
RET_19: 0.4980
RET_20: 0.5003
RET_ 1: same=0.5216, flip=0.4784
RET_ 2: same=0.4975, flip=0.5025
RET_ 3: same=0.4968, flip=0.5032
RET_ 4: same=0.4982, flip=0.5018
RET_ 5: same=0.4967, flip=0.5033
RET_ 6: same=0.5037, flip=0.4963
RET_ 7: same=0.5080, flip=0.4920
RET_ 8: same=0.5003, flip=0.4997
RET_ 9: same=0.5061, flip=0.4939
RET_10: same=0.5048, flip=0.4953
RET_11: same=0.5045, flip=0.4955
RET_12: same=0.5064, flip=0.4936
RET_13: same=0.5004, flip=0.4997
RET_14: same=0.4948, flip=0.5053
RET_15: same=0.4971, flip=0.5030
RET_16: same=0.4999, flip=0.5002
RET_17: same=0.5045, flip=0.4955
RET_18: same=0.4996, flip=0.5004
RET_19: same=0.4980, flip=0.5020
RET_20: same=0.5003, flip=0.4998


In [32]:
print(type(y_train))
print(y_train.shape)
print(y_train.head())
print(y_train.columns if hasattr(y_train, "columns") else "no columns")

<class 'pandas.DataFrame'>
(527073, 1)
          target
ROW_ID          
0       0.009210
1       0.000405
2       0.005967
3      -0.000542
4      -0.002579
Index(['target'], dtype='str')


In [33]:
val_df = X_train.loc[val_mask]
val_y = (y_train.loc[val_mask, 'target'] > 0).astype(int)

tmp = pd.DataFrame({
    'TS': val_df['TS'],
    'correct': (
        (val_df['RET_1'] > 0).astype(int).values
        == val_y.values
    )
})

ret1_ts_acc = tmp.groupby('TS')['correct'].mean()

print(ret1_ts_acc.describe())

print("\nFraction of TS where RET1 > 50%:",
      (ret1_ts_acc > 0.50).mean())

print("Fraction of TS where RET1 > 55%:",
      (ret1_ts_acc > 0.55).mean())

print("Fraction of TS where RET1 < 45%:",
      (ret1_ts_acc < 0.45).mean())


count    505.000000
mean       0.528130
std        0.091168
min        0.271429
25%        0.474638
50%        0.521739
75%        0.584615
max        0.885714
Name: correct, dtype: float64

Fraction of TS where RET1 > 50%: 0.6198019801980198
Fraction of TS where RET1 > 55%: 0.3801980198019802
Fraction of TS where RET1 < 45%: 0.1900990099009901


In [34]:
snapshot_ret1 = (
    X_train.loc[val_mask]
    .groupby('TS')
    .agg(
        RET1_STD=('RET_1', 'std'),
        RET1_MEAN=('RET_1', 'mean'),
        RET1_POSITIVE_RATE=('RET_1', lambda x: (x > 0).mean())
    )
)

snapshot_ret1['RET1_ACCURACY'] = ret1_ts_acc

print(
    snapshot_ret1[
        ['RET1_STD', 'RET1_MEAN', 'RET1_POSITIVE_RATE', 'RET1_ACCURACY']
    ].corr()['RET1_ACCURACY']
)

RET1_STD              0.094798
RET1_MEAN            -0.035205
RET1_POSITIVE_RATE   -0.054560
RET1_ACCURACY         1.000000
Name: RET1_ACCURACY, dtype: float64


In [35]:
tmp = pd.DataFrame({
    'ALLOCATION': X_train.loc[val_mask, 'ALLOCATION'].values,
    'correct': (
        (X_train.loc[val_mask, 'RET_1'].values > 0).astype(int)
        ==
        (y_train.loc[val_mask, 'target'].values > 0).astype(int)
    )
})

alloc_ret1_acc = tmp.groupby('ALLOCATION')['correct'].agg(
    ['mean', 'count']
)

print(alloc_ret1_acc['mean'].describe())

print("\n>55%:")
print((alloc_ret1_acc['mean'] > 0.55).mean())

print("\n<45%:")
print((alloc_ret1_acc['mean'] < 0.45).mean())

print("\nTop:")
print(alloc_ret1_acc.sort_values('mean', ascending=False).head(10))

print("\nBottom:")
print(alloc_ret1_acc.sort_values('mean').head(10))

count    278.000000
mean       0.522110
std        0.029772
min        0.442446
25%        0.503597
50%        0.522065
75%        0.539568
max        0.666667
Name: mean, dtype: float64

>55%:
0.1618705035971223

<45%:
0.007194244604316547

Top:
                    mean  count
ALLOCATION                     
ALLOCATION_14   0.666667      3
ALLOCATION_46   0.666667      3
ALLOCATION_277  0.589928    278
ALLOCATION_263  0.589928    278
ALLOCATION_163  0.582734    278
ALLOCATION_210  0.580902    377
ALLOCATION_168  0.578249    377
ALLOCATION_204  0.577002    487
ALLOCATION_98   0.576177    361
ALLOCATION_257  0.576177    361

Bottom:
                    mean  count
ALLOCATION                     
ALLOCATION_06   0.442446    278
ALLOCATION_231  0.448753    361
ALLOCATION_73   0.451524    361
ALLOCATION_246  0.457064    361
ALLOCATION_58   0.461538    377
ALLOCATION_186  0.462604    361
ALLOCATION_57   0.469496    377
ALLOCATION_52   0.472149    377
ALLOCATION_49   0.473684    361
ALLOCATI

In [36]:
train_part = X_train.loc[train_mask].copy()
val_part = X_train.loc[val_mask].copy()

train_target_sign = (
    y_train.loc[train_mask, 'target'] > 0
).astype(int)

tmp_train = pd.DataFrame({
    'ALLOCATION': train_part['ALLOCATION'].values,
    'RET1_POS': (train_part['RET_1'].values > 0).astype(int),
    'TARGET': train_target_sign.values
})

# P(target positive | RET1 positive) vs P(target positive | RET1 negative)
mapping = tmp_train.groupby(
    ['ALLOCATION', 'RET1_POS']
)['TARGET'].mean()

print(mapping.head(10))

ALLOCATION     RET1_POS
ALLOCATION_01  0           0.528600
               1           0.534056
ALLOCATION_02  0           0.514006
               1           0.520913
ALLOCATION_03  0           0.431239
               1           0.517730
ALLOCATION_04  0           0.473684
               1           0.531746
ALLOCATION_05  0           0.486376
               1           0.499350
Name: TARGET, dtype: float64


In [37]:
mapping_df = (
    tmp_train
    .groupby(['ALLOCATION', 'RET1_POS'])['TARGET']
    .mean()
    .unstack()
)

# For each allocation, choose whether RET1 positive or negative
# historically corresponds to higher target-positive probability.
alloc_rule = (mapping_df[1] > mapping_df[0]).astype(int)

print(alloc_rule.value_counts())
print(mapping_df.head(20))

1    234
0     44
Name: count, dtype: int64
RET1_POS               0         1
ALLOCATION                        
ALLOCATION_01   0.528600  0.534056
ALLOCATION_02   0.514006  0.520913
ALLOCATION_03   0.431239  0.517730
ALLOCATION_04   0.473684  0.531746
ALLOCATION_05   0.486376  0.499350
ALLOCATION_06   0.541339  0.558140
ALLOCATION_07   0.496392  0.585185
ALLOCATION_08   0.479603  0.506540
ALLOCATION_09   0.466736  0.501010
ALLOCATION_10   0.496337  0.528830
ALLOCATION_100  0.474061  0.540404
ALLOCATION_101  0.510395  0.520202
ALLOCATION_102  0.465267  0.456929
ALLOCATION_103  0.534447  0.527163
ALLOCATION_104  0.506680  0.524004
ALLOCATION_105  0.513944  0.505333
ALLOCATION_106  0.461942  0.511364
ALLOCATION_107  0.494229  0.531532
ALLOCATION_108  0.449875  0.520567
ALLOCATION_109  0.469643  0.539629


In [38]:
# Validation prediction using allocation-specific rule learned ONLY from train TSs

val_ret1_pos = (val_part['RET_1'].values > 0).astype(int)

# For each validation row:
# if allocation's learned rule = 1 -> predict RET1 sign
# if rule = 0 -> predict opposite of RET1 sign
alloc_rule_arr = (
    val_part['ALLOCATION']
    .map(alloc_rule)
    .fillna(1)
    .values
)

pred = np.where(
    alloc_rule_arr == 1,
    val_ret1_pos,
    1 - val_ret1_pos
)

actual = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

print("Allocation-specific RET1 rule:", accuracy_score(actual, pred))

# Baseline
print(
    "Plain RET1 sign:",
    accuracy_score(actual, val_ret1_pos)
)

Allocation-specific RET1 rule: 0.5147013278618714
Plain RET1 sign: 0.5215729162634044


In [39]:
val = X_train.loc[val_mask].copy()
actual = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

cs_median = val.groupby('TS')['RET_1'].transform('median')
relative_ret1 = val['RET_1'] - cs_median

pred = (relative_ret1 > 0).astype(int)

print(
    "RET1 - TS median:",
    accuracy_score(actual, pred)
)

print(
    "Plain RET1:",
    accuracy_score(actual, (val['RET_1'] > 0).astype(int))
)

RET1 - TS median: 0.5186887847934653
Plain RET1: 0.5215729162634044


In [40]:
# Get the 276-allocation snapshots
ts_counts = X_train.groupby('TS').size()

ts_276 = ts_counts[ts_counts == 276].index

# Split ONLY these snapshots
ts_276_train, ts_276_val = train_test_split(
    ts_276,
    test_size=0.20,
    random_state=42
)

train_mask_276 = X_train['TS'].isin(ts_276_train)
val_mask_276 = X_train['TS'].isin(ts_276_val)

print("Train TS:", train_mask_276.sum())
print("Val TS:", val_mask_276.sum())
print("Validation snapshots:", len(ts_276_val))

Train TS: 280968
Val TS: 70380
Validation snapshots: 255


In [41]:
print(len(ts_276))
print(len(ts_276_train))
print(len(ts_276_val))

1273
1018
255


In [42]:
val = X_train.loc[val_mask_276]
actual = (y_train.loc[val_mask_276, 'target'].values > 0).astype(int)

pred = (val['RET_1'].values > 0).astype(int)

print("276-config RET1 accuracy:", accuracy_score(actual, pred))

276-config RET1 accuracy: 0.5128303495311168


In [43]:
val_ts = X_train.loc[val_mask, 'TS']

is_276 = (
    X_train.loc[val_mask]
    .groupby('TS')['ALLOCATION']
    .transform('size')
    .values == 276
)

print(
    "Full grouped validation:",
    accuracy_score(
        y_train.loc[val_mask, 'target'].values > 0,
        val_pred
    )
)

print(
    "276 snapshots only:",
    accuracy_score(
        (y_train.loc[val_mask, 'target'].values > 0)[is_276],
        val_pred[is_276]
    )
)

Full grouped validation: 0.5192694824048624
276 snapshots only: 0.5135114734299517


In [44]:
presence = (
    X_train.pivot_table(
        index='TS',
        columns='ALLOCATION',
        values='RET_1',
        aggfunc='first'
    )
)

print("14 present:", presence['ALLOCATION_14'].notna().sum())
print("46 present:", presence['ALLOCATION_46'].notna().sum())

print("\nBoth present:",
      (presence['ALLOCATION_14'].notna() &
       presence['ALLOCATION_46'].notna()).sum())

print("\nSnapshot sizes when 14 is present:")
print(
    X_train[X_train['ALLOCATION'] == 'ALLOCATION_14']
    .groupby('TS').size()
    .value_counts()
    .sort_index()
)

14 present: 19
46 present: 19

Both present: 19

Snapshot sizes when 14 is present:
1    19
Name: count, dtype: int64


In [45]:
presence = (
    X_train.pivot_table(
        index='TS',
        columns='ALLOCATION',
        values='RET_1',
        aggfunc='size',
        fill_value=0
    )
)

config_counts = presence.astype(int).astype(str).agg(''.join, axis=1).value_counts()

print("Unique allocation configurations:", len(config_counts))
print("\nConfiguration frequencies:")
print(config_counts.head(30))

Unique allocation configurations: 24

Configuration frequencies:
11111111111111111111111111111111111111111111111111111011111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111011111111111111111111111111111111111111111111111111111    1273
01011011100101111110011111110101110111111101011110111000111111111111101111111010100011101101010011111111110111111101111011011011111111100001111111111101001110111111101101111101001011111111000011010111111101101101111111101100010111111011111111110011100111111101011111111001011110     304
00000000100101100100010100110100000100010000000010110000000001100001000101101000100000001001010000001100000000100100010000011000000011000000000101101000000000001010001000000001000010001001000000000000011000000000001001001000000000000010010101000000100000011000001100101001001100     265
00000001100101100100010100110100000100010000000010110000010001110001000101

In [46]:
# Configuration ID from allocation presence
config_key = (
    X_train.pivot_table(
        index='TS',
        columns='ALLOCATION',
        values='RET_1',
        aggfunc='size',
        fill_value=0
    )
    .astype(int)
    .astype(str)
    .agg(''.join, axis=1)
)

config_id = pd.factorize(config_key)[0]

ts_to_config = pd.Series(config_id, index=config_key.index, name='CONFIG_ID')

tmp = pd.DataFrame({
    'TS': X_train['TS'].values,
    'target': y_train['target'].values
})

tmp['CONFIG_ID'] = tmp['TS'].map(ts_to_config)

config_target = (
    tmp.groupby('CONFIG_ID')['target']
    .agg(['mean', 'count'])
    .sort_values('count', ascending=False)
)

print(config_target)

               mean   count
CONFIG_ID                  
0          0.000027  351348
4         -0.000009   62624
2          0.000148   35292
3          0.000005   17225
1         -0.000035   14697
13        -0.000045    9870
9          0.000206    6072
8         -0.000088    5768
10        -0.000017    5356
5          0.000383    3914
11         0.000398    3080
17         0.000165    2380
15         0.000019    1496
19        -0.000079    1400
21        -0.000543    1360
6          0.000066    1224
12        -0.000179    1050
22         0.001377     945
18         0.000586     680
14         0.000685     540
16        -0.001014     280
20        -0.000029     264
7          0.002150     138
23        -0.000791      70


In [47]:
tmp['TARGET_POSITIVE'] = (tmp['target'] > 0).astype(int)

config_hit_rate = (
    tmp.groupby('CONFIG_ID')['TARGET_POSITIVE']
    .agg(['mean', 'count'])
    .sort_values('count', ascending=False)
)

print(config_hit_rate)

               mean   count
CONFIG_ID                  
0          0.507457  351348
4          0.499154   62624
2          0.518361   35292
3          0.507518   17225
1          0.504457   14697
13         0.508713    9870
9          0.514328    6072
8          0.491505    5768
10         0.502240    5356
5          0.518651    3914
11         0.513636    3080
17         0.538655    2380
15         0.517380    1496
19         0.498571    1400
21         0.425735    1360
6          0.512255    1224
12         0.440000    1050
22         0.611640     945
18         0.555882     680
14         0.577778     540
16         0.346429     280
20         0.503788     264
7          0.688406     138
23         0.342857      70


In [48]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# One config per TS
ts_config = ts_to_config.rename('CONFIG_ID')

tmp = pd.DataFrame({
    'TS': X_train['TS'].values,
    'target': y_train['target'].values
})
tmp['CONFIG_ID'] = tmp['TS'].map(ts_config)
tmp['y'] = (tmp['target'] > 0).astype(int)

pred_parts = []

for config_id, g in tmp.groupby('CONFIG_ID'):
    ts = g['TS'].unique()

    # Need enough snapshots to make a meaningful split
    if len(ts) < 20:
        continue

    tr_ts, va_ts = train_test_split(
        ts,
        test_size=0.20,
        random_state=42
    )

    tr = g[g['TS'].isin(tr_ts)]
    va = g[g['TS'].isin(va_ts)]

    p = tr['y'].mean()
    va = va.copy()
    va['pred'] = int(p >= 0.5)

    pred_parts.append(va[['TS', 'y', 'pred']])

config_val = pd.concat(pred_parts, ignore_index=True)

accuracy = (config_val['y'] == config_val['pred']).mean()

print("Configuration-only accuracy:", accuracy)
print("Rows:", len(config_val))

Configuration-only accuracy: 0.5048538034403759
Rows: 103012


In [49]:
val = X_train_minimal.loc[val_mask].copy()
val['y'] = y_train.loc[val_mask, 'target'].values > 0

val['RET1_BIN'] = pd.qcut(
    val['RET_1'],
    q=10,
    duplicates='drop'
)

ret1_bins = (
    val.groupby('RET1_BIN', observed=True)
       .agg(
           RET1_MEAN=('RET_1', 'mean'),
           TARGET_RATE=('y', 'mean'),
           N=('y', 'size')
       )
)

print(ret1_bins)

                       RET1_MEAN  TARGET_RATE      N
RET1_BIN                                            
(-0.0404, -0.00312]    -0.005340     0.459886  10333
(-0.00312, -0.00187]   -0.002425     0.470286  10332
(-0.00187, -0.00109]   -0.001454     0.494193  10332
(-0.00109, -0.000474]  -0.000764     0.501694  10333
(-0.000474, 5.96e-05]  -0.000204     0.495741  10332
(5.96e-05, 0.000607]    0.000332     0.517906  10332
(0.000607, 0.00122]     0.000910     0.519501  10333
(0.00122, 0.00201]      0.001599     0.530972  10332
(0.00201, 0.00331]      0.002585     0.523713  10332
(0.00331, 0.111]        0.005507     0.550373  10333


In [50]:
val = X_train.loc[val_mask, ['TS', 'RET_1']].copy()
val['y'] = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

# Snapshot-level RET1 volatility
ret1_std = val.groupby('TS')['RET_1'].transform('std')

val['RET1_SCALED'] = val['RET_1'] / ret1_std

val['RET1_SCALED_BIN'] = pd.qcut(
    val['RET1_SCALED'],
    q=10,
    duplicates='drop'
)

scaled_bins = (
    val.groupby('RET1_SCALED_BIN', observed=True)
       .agg(
           RET1_SCALED_MEAN=('RET1_SCALED', 'mean'),
           TARGET_RATE=('y', 'mean'),
           N=('y', 'size')
       )
)

print(scaled_bins)

                  RET1_SCALED_MEAN  TARGET_RATE      N
RET1_SCALED_BIN                                       
(-8.094, -1.177]         -1.814518     0.462789  10333
(-1.177, -0.714]         -0.921937     0.467964  10332
(-0.714, -0.414]         -0.557726     0.492547  10332
(-0.414, -0.182]         -0.295565     0.503532  10333
(-0.182, 0.0234]         -0.078692     0.494967  10332
(0.0234, 0.236]           0.128928     0.521099  10332
(0.236, 0.472]            0.350437     0.521436  10333
(0.472, 0.775]            0.615676     0.532424  10332
(0.775, 1.24]             0.986892     0.526229  10332
(1.24, 14.892]            1.881782     0.541276  10333


In [51]:
val = X_train.loc[val_mask, ['TS', 'RET_1']].copy()
val['y'] = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

# Cross-sectional percentile rank of RET1
val['RET1_RANK'] = (
    val.groupby('TS')['RET_1']
       .rank(method='average', pct=True)
)

val['RET1_RANK_BIN'] = pd.qcut(
    val['RET1_RANK'],
    q=10,
    duplicates='drop'
)

rank_bins = (
    val.groupby('RET1_RANK_BIN', observed=True)
       .agg(
           RET1_RANK_MEAN=('RET1_RANK', 'mean'),
           TARGET_RATE=('y', 'mean'),
           N=('y', 'size')
       )
)

print(rank_bins)

                  RET1_RANK_MEAN  TARGET_RATE      N
RET1_RANK_BIN                                       
(0.00262, 0.101]        0.052479     0.460340  10338
(0.101, 0.203]          0.153408     0.483224  10521
(0.203, 0.304]          0.253525     0.491149  10168
(0.304, 0.402]          0.353063     0.498462  10400
(0.402, 0.504]          0.453820     0.505661  10422
(0.504, 0.601]          0.553426     0.512548  10161
(0.601, 0.703]          0.653197     0.511761  10458
(0.703, 0.801]          0.753307     0.531528  10229
(0.801, 0.902]          0.853054     0.534001  10382
(0.902, 1.0]            0.952866     0.536262  10245


In [52]:
val = X_train.loc[val_mask, ['TS', 'RET_1']].copy()
val['y'] = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

val['RET1_RANK'] = (
    val.groupby('TS')['RET_1']
       .rank(method='average', pct=True)
)

# Predict positive when RET1 rank is above the median
val['pred'] = (val['RET1_RANK'] > 0.5).astype(int)

accuracy = (val['pred'] == val['y']).mean()

print("RET1 cross-sectional rank accuracy:", accuracy)

RET1 cross-sectional rank accuracy: 0.5186887847934653


In [53]:
val = X_train.loc[val_mask, ['TS', 'RET_1']].copy()
val['y'] = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

val['RET1_RANK'] = (
    val.groupby('TS')['RET_1']
       .rank(method='average', pct=True)
)

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (val['RET1_RANK'] > threshold).astype(int)
    acc = (pred == val['y']).mean()
    print(f"threshold={threshold:.1f}: {acc:.6f}")

threshold=0.1: 0.514188
threshold=0.2: 0.517605
threshold=0.3: 0.519453
threshold=0.4: 0.519811
threshold=0.5: 0.518689
threshold=0.6: 0.516463
threshold=0.7: 0.513927
threshold=0.8: 0.507810
threshold=0.9: 0.500842


In [54]:
from sklearn.linear_model import LogisticRegression

rank_cols = []

X_rank = X_train.loc[:, ['TS'] + [f'RET_{i}' for i in range(1, 21)]].copy()

for i in range(1, 21):
    col = f'RET_{i}'
    rank_col = f'{col}_RANK'

    X_rank[rank_col] = (
        X_rank.groupby('TS')[col]
              .rank(method='average', pct=True)
              .fillna(0.5)
    )

    rank_cols.append(rank_col)

Xr_tr = X_rank.loc[train_mask, rank_cols]
Xr_val = X_rank.loc[val_mask, rank_cols]

yr_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
yr_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

model = LogisticRegression(
    max_iter=1000,
    C=1.0
)

model.fit(Xr_tr, yr_tr)

pred = model.predict(Xr_val)

print("Rank-only logistic accuracy:", (pred == yr_val).mean())

Rank-only logistic accuracy: 0.5174499632224846


In [55]:
from sklearn.linear_model import LogisticRegression

X_lr = X_train.loc[:, ['TS', 'RET_1']].copy()

X_lr['RET1_RANK'] = (
    X_lr.groupby('TS')['RET_1']
        .rank(method='average', pct=True)
        .fillna(0.5)
)

X_lr['RET_1'] = X_lr['RET_1'].fillna(0)

X_tr = X_lr.loc[train_mask, ['RET_1', 'RET1_RANK']]
X_val = X_lr.loc[val_mask, ['RET_1', 'RET1_RANK']]

y_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
y_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

model = LogisticRegression(max_iter=1000, C=1.0)
model.fit(X_tr, y_tr)

pred = model.predict(X_val)

print("RET1 + rank logistic accuracy:", (pred == y_val).mean())

RET1 + rank logistic accuracy: 0.519898571483876


In [56]:
from sklearn.ensemble import ExtraTreesClassifier

X_tr = X_train.loc[train_mask, ['RET_1']].fillna(0)
X_val = X_train.loc[val_mask, ['RET_1']].fillna(0)

y_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
y_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

model = ExtraTreesClassifier(
    n_estimators=500,
    max_features=1.0,
    min_samples_leaf=200,
    n_jobs=-1,
    random_state=42
)

model.fit(X_tr, y_tr)
pred = model.predict(X_val)

print("RET1-only ExtraTrees accuracy:", (pred == y_val).mean())

RET1-only ExtraTrees accuracy: 0.5066393093569742


In [57]:
from sklearn.linear_model import LogisticRegression

ret_cols = [f'RET_{i}' for i in range(1, 21)]

X_sign = X_train.loc[:, ret_cols].copy()

# Convert each lag to its sign: -1, 0, +1
X_sign = np.sign(X_sign).fillna(0)

X_tr = X_sign.loc[train_mask]
X_val = X_sign.loc[val_mask]

y_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
y_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

model = LogisticRegression(
    max_iter=1000,
    C=1.0
)

model.fit(X_tr, y_tr)

pred = model.predict(X_val)

print("20 return signs logistic accuracy:", (pred == y_val).mean())

20 return signs logistic accuracy: 0.5195501529170377


In [58]:
from sklearn.linear_model import LogisticRegression

X_mag = X_train.loc[:, ['RET_1']].copy()
X_mag['RET1_SIGN'] = np.sign(X_mag['RET_1']).fillna(0)
X_mag['RET1_ABS'] = X_mag['RET_1'].abs().fillna(0)

X_tr = X_mag.loc[train_mask, ['RET1_SIGN', 'RET1_ABS']]
X_val = X_mag.loc[val_mask, ['RET1_SIGN', 'RET1_ABS']]

y_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
y_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

model = LogisticRegression(
    max_iter=1000,
    C=1.0
)

model.fit(X_tr, y_tr)

pred = model.predict(X_val)

print("RET1 sign + magnitude logistic accuracy:", (pred == y_val).mean())

RET1 sign + magnitude logistic accuracy: 0.5215729162634044


In [59]:
val = X_train.loc[val_mask, ['TS', 'RET_1']].copy()
val['y'] = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

val['ret1_pred'] = (val['RET_1'] > 0).astype(int)

ts_stats = (
    val.groupby('TS')
       .agg(
           RET1_MEAN=('RET_1', 'mean'),
           RET1_STD=('RET_1', 'std'),
           RET1_POS_RATE=('RET_1', lambda x: (x > 0).mean()),
           ACC=('ret1_pred', lambda x: np.nan)
       )
)

# Calculate actual per-TS accuracy
ts_acc = (
    val.groupby('TS')
       .apply(lambda g: (g['ret1_pred'] == g['y']).mean())
       .rename('ACC')
)

ts_stats['ACC'] = ts_acc

# Split snapshots into 5 groups by cross-sectional RET1 positive rate
ts_stats['REGIME'] = pd.qcut(
    ts_stats['RET1_POS_RATE'],
    5,
    duplicates='drop'
)

print(
    ts_stats.groupby('REGIME', observed=True)
            .agg(
                accuracy=('ACC', 'mean'),
                n_ts=('ACC', 'size'),
                ret1_pos_rate=('RET1_POS_RATE', 'mean')
            )
)

                accuracy  n_ts  ret1_pos_rate
REGIME                                       
(0.153, 0.432]  0.538802   103       0.370704
(0.432, 0.488]  0.522896    99       0.459140
(0.488, 0.538]  0.524221   103       0.513634
(0.538, 0.586]  0.518878    99       0.562041
(0.586, 0.815]  0.535432   101       0.639679


In [60]:
val = X_train.loc[val_mask, ['TS', 'RET_1']].copy()
val['y'] = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

# Snapshot-level positive RET1 rate
pos_rate = (
    val.groupby('TS')['RET_1']
       .apply(lambda x: (x > 0).mean())
)

# Quintile boundaries
q20 = pos_rate.quantile(0.20)
q80 = pos_rate.quantile(0.80)

val['POS_RATE'] = val['TS'].map(pos_rate)

# Baseline
val['pred_base'] = (val['RET_1'] > 0).astype(int)

# Flip RET1 only in the middle 60%
val['pred_regime'] = val['pred_base']

middle = val['POS_RATE'].between(q20, q80)
val.loc[middle, 'pred_regime'] = 1 - val.loc[middle, 'pred_regime']

print("Baseline:", (val['pred_base'] == val['y']).mean())
print("Regime rule:", (val['pred_regime'] == val['y']).mean())

print("\nBy regime:")
print(
    val.groupby(
        pd.cut(
            val['POS_RATE'],
            bins=[-np.inf, q20, q80, np.inf]
        )
    ).apply(
        lambda g: (g['pred_regime'] == g['y']).mean(),
        include_groups=False
    )
)

Baseline: 0.5215729162634044
Regime rule: 0.5000871046417096

By regime:
POS_RATE
(-inf, 0.432]     0.530284
(0.432, 0.586]    0.482324
(0.586, inf]      0.527436
dtype: float64


In [61]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

df = X_train.loc[:, ['ALLOCATION', 'RET_1']].copy()
df['RET1_SIGN'] = np.sign(df['RET_1']).fillna(0)

X_tr = df.loc[train_mask, ['ALLOCATION', 'RET1_SIGN']]
X_val = df.loc[val_mask, ['ALLOCATION', 'RET1_SIGN']]

y_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
y_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

pre = ColumnTransformer([
    ('alloc', OneHotEncoder(handle_unknown='ignore'), ['ALLOCATION']),
    ('num', 'passthrough', ['RET1_SIGN'])
])

model = make_pipeline(
    pre,
    LogisticRegression(
        C=0.01,
        max_iter=1000
    )
)

model.fit(X_tr, y_tr)
pred = model.predict(X_val)

print("Allocation + RET1 sign:", (pred == y_val).mean())

Allocation + RET1 sign: 0.5239731330571794


In [62]:
for C in [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]:
    model = make_pipeline(
        pre,
        LogisticRegression(
            C=C,
            max_iter=1000
        )
    )

    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)

    print(C, (pred == y_val).mean())

0.001 0.5242731601564012
0.003 0.5254926251403352
0.01 0.5239731330571794
0.03 0.5232762959235028
0.1 0.523489218381015
0.3 0.523489218381015
1.0 0.5233924354457822


In [63]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from scipy import sparse

df = X_train.loc[:, ['ALLOCATION', 'RET_1']].copy()
df['RET1_SIGN'] = np.sign(df['RET_1']).fillna(0)

alloc = pd.get_dummies(df['ALLOCATION'], dtype=float).values
ret_sign = df['RET1_SIGN'].values.reshape(-1, 1)

# Allocation-specific RET1 response:
# each allocation gets its own RET1_SIGN coefficient
interaction = alloc * ret_sign

# Keep the global RET1 sign too
X = np.hstack([
    alloc,
    ret_sign,
    interaction
])

X_tr = X[train_mask]
X_val = X[val_mask]

y_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
y_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

for C in [0.0001, 0.0003, 0.001, 0.003, 0.01]:
    model = LogisticRegression(
        C=C,
        max_iter=1000
    )

    model.fit(X_tr, y_tr)
    pred = model.predict(X_val)

    print(C, (pred == y_val).mean())

0.0001 0.5215729162634044
0.0003 0.5225697804963029
0.001 0.5242925167434478
0.003 0.5237505323061438
0.01 0.5214083852735085


In [64]:
# Training-only allocation target rate
tmp = pd.DataFrame({
    'ALLOCATION': X_train.loc[train_mask, 'ALLOCATION'].values,
    'y': (y_train.loc[train_mask, 'target'].values > 0).astype(int)
})

alloc_rate = tmp.groupby('ALLOCATION')['y'].mean()

# Map to both train and validation
df = X_train.loc[:, ['ALLOCATION', 'RET_1']].copy()

df['ALLOC_TARGET_RATE'] = (
    df['ALLOCATION']
    .map(alloc_rate)
    .fillna(tmp['y'].mean())
)

df['RET1_SIGN'] = np.sign(df['RET_1']).fillna(0)

X_tr = df.loc[train_mask, ['ALLOC_TARGET_RATE', 'RET1_SIGN']]
X_val = df.loc[val_mask, ['ALLOC_TARGET_RATE', 'RET1_SIGN']]

y_tr = (y_train.loc[train_mask, 'target'].values > 0).astype(int)
y_val = (y_train.loc[val_mask, 'target'].values > 0).astype(int)

for C in [0.01, 0.1, 1.0]:
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_tr, y_tr)

    pred = model.predict(X_val)

    print(C, (pred == y_val).mean())

0.01 0.5265475591343735
0.1 0.5233537222716891
1.0 0.5235666447292013


In [65]:
overall_rate = tmp['y'].mean()

df = X_train.loc[:, ['ALLOCATION', 'RET_1']].copy()

df['ALLOC_TARGET_RATE'] = (
    df['ALLOCATION']
    .map(alloc_rate)
    .fillna(overall_rate)
)

# Center around the unconditional 50% direction
df['ALLOC_EDGE'] = df['ALLOC_TARGET_RATE'] - overall_rate

df['RET1_SIGN'] = np.sign(df['RET_1']).fillna(0)

X_tr = df.loc[train_mask, ['ALLOC_EDGE', 'RET1_SIGN']]
X_val = df.loc[val_mask, ['ALLOC_EDGE', 'RET1_SIGN']]

for C in [0.001, 0.003, 0.01, 0.03, 0.1]:
    model = LogisticRegression(C=C, max_iter=1000)
    model.fit(X_tr, y_tr)

    pred = model.predict(X_val)

    print(C, (pred == y_val).mean())

0.001 0.5215729162634044
0.003 0.5232956525105493
0.01 0.5265475591343735
0.03 0.5247667531260888
0.1 0.5233537222716891


In [66]:
# Use the original training row order
df = X_train[['ALLOCATION']].copy()
df['y'] = (y_train['target'].values > 0).astype(int)

# First 70% of training observations vs last 30%
cut = int(len(df) * 0.70)

early = df.iloc[:cut]
late  = df.iloc[cut:]

early_rate = early.groupby('ALLOCATION')['y'].mean()
late_rate  = late.groupby('ALLOCATION')['y'].mean()

persistence = pd.DataFrame({
    'early_rate': early_rate,
    'late_rate': late_rate
}).dropna()

print("Allocation correlation:",
      persistence['early_rate'].corr(persistence['late_rate']))

print("\nMean absolute change:",
      (persistence['early_rate'] - persistence['late_rate']).abs().mean())

print("\nEarly rate:")
print(persistence['early_rate'].describe())

print("\nLate rate:")
print(persistence['late_rate'].describe())

Allocation correlation: 0.6590847150232134

Mean absolute change: 0.020132794225828206

Early rate:
count    278.000000
mean       0.504431
std        0.028289
min        0.388676
25%        0.488453
50%        0.503336
75%        0.517976
max        0.654343
Name: early_rate, dtype: float64

Late rate:
count    278.000000
mean       0.509653
std        0.032306
min        0.375000
25%        0.494917
50%        0.510039
75%        0.526386
max        0.669388
Name: late_rate, dtype: float64


In [67]:
overall_rate = tmp['y'].mean()

alloc_stats = (
    tmp.groupby('ALLOCATION')['y']
       .agg(['mean', 'count'])
)

df = X_train.loc[:, ['ALLOCATION', 'RET_1']].copy()

for k in [10, 50, 100, 250, 500, 1000, 2000]:
    shrunk = (
        (alloc_stats['count'] * alloc_stats['mean'] +
         k * overall_rate)
        /
        (alloc_stats['count'] + k)
    )

    df['ALLOC_RATE'] = (
        df['ALLOCATION']
        .map(shrunk)
        .fillna(overall_rate)
    )

    df['RET1_SIGN'] = np.sign(df['RET_1']).fillna(0)

    X_tr = df.loc[train_mask, ['ALLOC_RATE', 'RET1_SIGN']]
    X_val = df.loc[val_mask, ['ALLOC_RATE', 'RET1_SIGN']]

    model = LogisticRegression(C=0.01, max_iter=1000)
    model.fit(X_tr, y_tr)

    pred = model.predict(X_val)

    print(k, (pred == y_val).mean())

10 0.5265475591343735
50 0.5265475591343735
100 0.5262378537416282
250 0.5262088188610584
500 0.5259958964035462
1000 0.5238473152413766
2000 0.523218226162363


In [68]:
df = X_train.loc[:, ['TS', 'ALLOCATION', 'RET_1']].copy()

df['RET1_SIGN'] = np.sign(df['RET_1']).fillna(0)

# Snapshot-level current market state
df['TS_RET1_POS_RATE'] = (
    df.groupby('TS')['RET_1']
      .transform(lambda x: (x > 0).mean())
)

# Historical allocation propensity
df['ALLOC_RATE'] = (
    df['ALLOCATION']
    .map(alloc_rate)
    .fillna(overall_rate)
)

X_tr = df.loc[
    train_mask,
    ['ALLOC_RATE', 'RET1_SIGN', 'TS_RET1_POS_RATE']
]

X_val = df.loc[
    val_mask,
    ['ALLOC_RATE', 'RET1_SIGN', 'TS_RET1_POS_RATE']
]

model = LogisticRegression(
    C=0.01,
    max_iter=1000
)

model.fit(X_tr, y_tr)
pred = model.predict(X_val)

print(
    "Allocation rate + RET1 sign + market state:",
    (pred == y_val).mean()
)

Allocation rate + RET1 sign + market state: 0.526092679338779


In [69]:
tmp = X_train[['ALLOCATION']].copy()
tmp['y'] = (y_train['target'] > 0).astype(int).values

mid = len(tmp) // 2

early = (
    tmp.iloc[:mid]
       .groupby('ALLOCATION')['y']
       .agg(['mean', 'count'])
       .rename(columns={'mean': 'EARLY_RATE'})
)

late = (
    tmp.iloc[mid:]
       .groupby('ALLOCATION')['y']
       .mean()
       .rename('LATE_RATE')
)

persist = early.join(late, how='inner')

print("Correlation:", persist['EARLY_RATE'].corr(persist['LATE_RATE']))
print("Mean absolute change:",
      (persist['EARLY_RATE'] - persist['LATE_RATE']).abs().mean())

Correlation: 0.7111236148240392
Mean absolute change: 0.018257895908146256


In [70]:
df = X_train[['ALLOCATION', 'RET_1']].copy()

df['RET1_SIGN'] = np.sign(df['RET_1']).fillna(0)

# Historical mean RET1 for each allocation, using training rows only
alloc_ret_mean = (
    X_train.loc[train_mask, ['ALLOCATION', 'RET_1']]
    .groupby('ALLOCATION')['RET_1']
    .mean()
)

df['ALLOC_RET_MEAN'] = (
    df['ALLOCATION']
    .map(alloc_ret_mean)
    .fillna(X_train.loc[train_mask, 'RET_1'].mean())
)

df['RET1_MINUS_ALLOC_MEAN'] = (
    df['RET_1'] - df['ALLOC_RET_MEAN']
)

df['ALLOC_RATE'] = (
    df['ALLOCATION']
    .map(alloc_rate)
    .fillna(overall_rate)
)

features = [
    'ALLOC_RATE',
    'RET1_SIGN',
    'RET1_MINUS_ALLOC_MEAN'
]

X_tr = df.loc[train_mask, features]
X_val = df.loc[val_mask, features]

model = LogisticRegression(C=0.01, max_iter=1000)
model.fit(X_tr, y_tr)

pred = model.predict(X_val)

print((pred == y_val).mean())

0.5265475591343735


In [71]:
X_tr = X_train.loc[train_mask, ['ALLOCATION']]
X_val = X_train.loc[val_mask, ['ALLOCATION']]

model = LogisticRegression(
    C=0.003,
    max_iter=1000
)

model.fit(
    pd.get_dummies(X_tr),
    y_tr
)

val_encoded = pd.get_dummies(X_val)

# Make columns identical
val_encoded = val_encoded.reindex(
    columns=pd.get_dummies(X_tr).columns,
    fill_value=0
)

pred = model.predict(val_encoded)

print((pred == y_val).mean())

0.5131624791916689
